# Entregável 1 — Especificação e Baseline

> **Grupo:** Rodolfo Dalla Costa, Thais Caroline Murer, Werner Conrado Jacob Denzin<br>
> **Tema/Projeto:** Sistema multiagente para suporte à perguntas (em linguagem natural) sobre dados educacionais brasileiros<br>
> **Disciplina:** INF0093 — Projeto Prático com Sistemas Multiagentes — 2S/2026<br>
> **Data:** 07/09/2026<br>

Este notebook deve conter a **especificação inicial do sistema**, a implementação de um
**baseline funcional**, casos de teste e uma análise crítica das limitações observadas.

Use células **Markdown** para documentação, justificativas e análise.
Use células **Python** para configuração, implementação, experimentos, testes e coleta de resultados.

**Antes de enviar:** execute o notebook do início ao fim (`Runtime → Run all`) e salve com as
saídas visíveis. Um notebook sem saídas não permite avaliar o baseline.


# 1. Descrição do problema

* O Brasil publica microdados educacionais e socioeconômicos de altíssima qualidade, sob licença aberta —
como ENEM (INEP/MEC) e os agregados municipais do IBGE. Porém, na prática, para interpretá-los 
de modo efetivo, contribuindo, principalmente, para tomadas de decisão, há a necessidade de conhecimento 
técnico especializado — por exemplo: os microdados do ENEM 2023 correspondem a um CSV de 1,8 GB com 76 colunas 
codificadas e o cruzamento com os dados do IBGE exige conhecimento adicional sobre APIs específicas que 
viabilizem tal agregação/correlação.

* Uma pergunta, por exemplo, do secretário municipal de educação —
*"a média de matemática da minha cidade está acima ou abaixo da média do estado?"* —
exigiria hoje horas de trabalho: _(a) download dos dados; (b) compreender o dicionário de variáveis; (c) selecionar as variáveis relevantes para esta pergunta; (d)escrever a `query/groupby`; (e) achar o código IBGE do município; e (f) interpretar o resultado_. **O dado é aberto, mas não é acessível.**

* **Relevância**

  * Quem toma decisão sobre educação municipal — secretarias, conselhos, imprensa local,
ONGs — raramente possui uma equipe de Ciência de Dados — cenário que influencia diretamente a qualidade e/ou viabilidade para desenvolvimento de análises específicas, customizadas e/ou particulares sobre um determinado domínio, recorrendo, portanto, à estudos e/ou rankings terceiros publicados anualmente.

* **Contexto**

  * Uso analítico e exploratório para a realização de perguntas concretas e específicas sobre um
município, um estado ou uma região, com a garantia de obter resultados acurados, seguros para tomadas
 de decisão e planejamento educacional.

* **Objetivo do Sistema**

  * Prover uma interface, através da qual usuários possam utilizá-la para realizar perguntas em linguagem natural (Português) sobre a qualidade e desempenho educacional (e socioeconômico) a partir de fontes de dados confiáveis, como ENEM e IBGE. Um sistema totalmente transparente e que abstrai qualquer necessidade de  conhecimento sobre as bases de dados e/ou técnico analítico.

* **Escopo desta 1a versão**

  * Esta versão tem por objetivo: _(a) receber uma pergunta do usuário; (b) interpretá-la; (c) consultar os dados necessários para respondê-la (sobre bases do ENEM 2023 e IBGE) e (d) gerar a resposta_. Itens que estão fora do escopo - gráficos, validação dos resultados, memória e arquitetura multiagente - aprimoramentos previstos para as próximas versões (roadmap).  
  

# 2. Usuário-alvo e stakeholders

| Usuário | Categoria | Objetivos |
|---|---|---|
| Analista educacional (secretaria municipal/estadual, jornalismo de dados, terceiro setor) | Principal | Obter, sob prazo, um número confiável sobre um município ou recorte geográfico, sem depender de um programador |
| Pesquisador em educação | Secundário | Definir hipóteses, verificar, validar resultados/dados, iterar com novas hipóteses |
| Gestor escolar / conselho municipal | Secundário | Comparar o município com vizinhos e com a média estadual, não apenas obter valores isolados |
| Jornalista de educação | Secundário | Verificar uma afirmação de fonte oficial, conhecendo o recorte exato para evitar publicações incorretas |
| INEP/MEC e IBGE | Stakeholder | Garantir que o dado seja usado sem ser deturpado; toda resposta do sistema precisa declarar o recorte aplicado |
| Secretarias de educação | Stakeholder | Consumir números confiáveis para publicação — um número errado tem custo político real |
| Sociedade / imprensa | Stakeholder | Beneficiar-se, indiretamente, de um dado público mais acessível |

# 3. Casos de uso principais

### UC-01 — Consulta agregada simples

| | |
|---|---|
| **Ator** | Analista de secretaria municipal |
| **Entrada** | *"Qual é a média geral do ENEM 2023 em Campinas?"* |
| **Objetivo** | Obter um valor único sobre um município |
| **Saída esperada** | Frase em português com o valor, o código pandas executado e a nota de recorte |
| **Sucesso** | O valor bate com a consulta de referência (tolerância de 1%) e o texto contém esse valor |

### UC-02 — Ranking com filtro

| | |
|---|---|
| **Ator** | Jornalista de dados |
| **Entrada** | *"Quais são os 5 municípios de SP com maior média em matemática entre os que têm ao menos 100 participantes?"* |
| **Objetivo** | Obter uma lista ordenada, com o filtro de robustez respeitado |
| **Saída esperada** | Lista dos municípios com o valor de cada um |
| **Sucesso** | Os cinco municípios de referência aparecem na resposta, e o filtro `n_participantes >= 100` foi aplicado |

### UC-03 — Cruzamento ENEM × IBGE

| | |
|---|---|
| **Ator** | Pesquisador em educação |
| **Entrada** | *"Qual é a correlação entre o PIB per capita e a média geral dos municípios com pelo menos 50 participantes?"* |
| **Objetivo** | Relacionar desempenho educacional e indicador socioeconômico |
| **Saída esperada** | Coeficiente de correlação e a ressalva de que ENEM é de 2023 e IBGE de 2021 |
| **Sucesso** | O coeficiente bate com a referência e a resposta menciona a diferença de anos |

### UC-04 — Pergunta que os dados não respondem

| | |
|---|---|
| **Ator** | Qualquer usuário |
| **Entrada** | *"Qual é o IDH de Recife?"* |
| **Objetivo** | Não receber um número inventado |
| **Saída esperada** | Recusa explícita, dizendo qual informação falta no recorte |
| **Sucesso** | O sistema se abstém, não executa código e nomeia a coluna ausente |

> UC-04 é o caso de uso mais importante para a credibilidade do sistema: um assistente que
> responde tudo é indistinguível de um que inventa. Ele é medido explicitamente (RF-05).


# 4. Escopo, não-objetivos e premissas

## Escopo desta versão

| | |
|---|---|
| **Dados** | ENEM 2023 agregado por município da escola × população e PIB municipais do IBGE (2021) — 5.481 municípios, 721.429 participantes, 15 colunas |
| **Interação** | uma pergunta por vez, em português, sem histórico (*stateless*) |
| **Saída** | texto em português + valor numérico/tabela + código pandas executado + nota de limitação |
| **Arquitetura** | uma chamada ao LLM que planeja a consulta; execução e formatação determinísticas |

## Não-objetivos (deliberadamente fora)

| Fora do escopo | Por quê |
|---|---|
| Geração de gráficos | é o agente *Visualizador*, previsto para o Entregável 3; embutir agora impede medir o ganho que ele traz |
| Validação automática do resultado com *retry* | é o agente *Validador* (Entregável 2); o baseline **precisa** errar de forma observável para que o ganho seja mensurável |
| Arquitetura multiagente (LangGraph) | o enunciado pede a solução mais simples adequada; complexidade sem necessidade demonstrada é justamente o que o curso pede para evitar |
| Memória de sessão / perguntas de acompanhamento | *stateless* é mais simples de avaliar; entra no Entregável 4 se houver folga |
| *Sandbox* real de execução | Entregável 2; aqui há apenas uma lista de tokens proibidos, e isso está registrado como limitação, não como solução |
| Microdados individuais (nota por candidato, questionário socioeconômico) | 1,8 GB não cabem no fluxo interativo; o pré-processamento é feito **uma vez**, fora do sistema |
| Séries temporais (ENEM de vários anos) | multiplica o ETL sem acrescentar nada à discussão arquitetural |
| Municípios de outros países, outros exames, outras bases | recorte de cinco semanas |
| Interface web / API pública | o notebook é a interface desta etapa |

## Premissas

1. O artefato pré-processado (`enem2023_ibge_municipios.csv`, 0,6 MB) cabe inteiro em
   memória e seu esquema cabe na janela de contexto do modelo — logo o LLM não precisa
   inspecionar os dados, apenas o esquema.
2. O usuário pergunta em português e sobre o recorte descrito; perguntas fora do recorte
   devem gerar abstenção, não erro.
3. O código de município do INEP (`CO_MUNICIPIO_ESC`) corresponde 1:1 ao código do IBGE —
   **verificado no ETL**: 5.481 de 5.481 municípios casaram.
4. O serviço de inferência (Groq) está disponível e `temperature=0` reduz — mas não
   elimina — a variação entre execuções.


# 5. Entradas e saídas

## Entradas

Uma **string em português**: pergunta em linguagem natural sobre o recorte ENEM 2023 ×
IBGE. Exemplos representativos:

- `"Qual é a média geral do ENEM 2023 em Campinas?"` — valor único
- `"Quantos municípios têm média geral acima de 550?"` — contagem com filtro
- `"Qual região tem a maior média de redação?"` — agregação por grupo
- `"Qual é a correlação entre PIB per capita e média geral?"` — cruzamento ENEM × IBGE
- `"Qual é a nota média de inglês em Salvador?"` — **fora do recorte**, deve gerar abstenção

Não há upload de arquivo: os dados são fixos e conhecidos pelo sistema.

## Saídas

Um objeto estruturado (validado por Pydantic) com:

| Campo | Tipo | Conteúdo |
|---|---|---|
| `texto` | str | Resposta em português, com o valor já formatado, seguida da nota de recorte |
| `resultado` | escalar / Series / DataFrame | Valor bruto devolvido pela execução do código |
| `codigo` | str | Expressão pandas efetivamente executada (evidência auditável) |
| `viavel` | bool | `False` quando a pergunta não é respondível com as colunas disponíveis |
| `motivo` | str | Estratégia adotada, ou o que falta nos dados quando `viavel=False` |
| `erro_execucao` | str \| None | Mensagem de erro, quando o código gerado falha |

Acompanham a saída as **métricas de execução** (latência, tokens de entrada e saída,
número de chamadas ao LLM), exigidas pela seção 3.2 do enunciado.


# 6. Requisitos funcionais

Cada requisito abaixo passa no teste *"consigo escrever hoje o critério que decide se ele
foi atendido?"*. A coluna **Como será verificado** aponta para a seção 15 (implementação
da verificação) e para os casos da seção 14.

| ID | Requisito | Como será verificado |
|---|---|---|
| **RF-01** | Responder perguntas de **valor único** (média, contagem, máximo) sobre o recorte: em 11 casos automáticos, acertar ao menos 8. | Comparação numérica do resultado executado com a resposta de referência, tolerância relativa de 1% (`resultado_correto`, casos T01, T02, T05) |
| **RF-02** | Responder perguntas de **ranking**, respeitando filtros de robustez declarados na pergunta (ex.: `n_participantes >= 100`). | Os nomes de referência aparecem na resposta final; o código gerado contém o filtro (`cobertura_lista`, caso T03) |
| **RF-03** | Responder perguntas que **cruzam ENEM e IBGE** (correlação, comparação entre grupos, recorte por população). | Comparação numérica com a referência (casos T06, T07, T08) |
| **RF-04** | Não produzir número que não venha da execução sobre o `DataFrame`: nenhum valor pode vir do conhecimento paramétrico do modelo. | Toda resposta aprovada precisa ter `codigo` não vazio e execução sem erro; números no texto que não aparecem no resultado reprovam (`texto_fiel`) |
| **RF-05** | **Abster-se** quando a pergunta exigir informação ausente do recorte (coluna inexistente, outro ano, outro exame), nomeando o que falta. | `viavel == False`, nenhum código executado e presença de marcador de ausência no texto (`abstencao_valida`, casos T09–T11) |
| **RF-06** | Expor como **evidência** o código pandas executado e o resultado bruto. | Presença não vazia de `codigo` e `resultado` em toda resposta viável (verificação estrutural na seção 16) |
| **RF-07** | Declarar as **limitações do recorte** (subconjunto de candidatos, anos diferentes entre ENEM e IBGE) junto da resposta. | O texto final contém a nota de recorte; ver limitação registrada na seção 18 — no baseline a nota é **fixa**, não sensível à pergunta |
| **RF-08** | Gerar **gráfico** quando o usuário pedir. | **Não atendido nesta versão** (não-objetivo declarado na seção 4); entra no Entregável 3 |
| **RF-09** | Pedir esclarecimento diante de pergunta **ambígua** (ex.: "média de São Paulo" — capital ou estado?). | Rubrica manual, casos T12 e T13; sem verificação automática nesta versão |

> RF-08 e RF-09 estão escritos aqui **de propósito**, mesmo sem serem atendidos: são a
> régua que tornará mensurável o ganho das próximas arquiteturas. Um requisito omitido não
> pode aparecer como melhoria no Entregável 4.


# 7. Requisitos não funcionais e restrições

| ID | Requisito | Valor de referência | Como será verificado |
|---|---|---|---|
| **RNF-01** | Saída estruturada e validada | esquema Pydantic; 0 erros de *parsing* nos 13 casos | campo `erro_parse` registrado por caso na seção 16 |
| **RNF-02** | Latência por consulta | mediana **< 10 s** | `latencia_s` medida por caso; mediana na seção 17 |
| **RNF-03** | Custo de chamadas ao LLM | **≤ 1 chamada** por pergunta no baseline; custo total do conjunto **< US$ 0,05** | `chamadas_llm` e contagem de tokens (`usage_metadata`) × preço da Groq |
| **RNF-04** | Reprodutibilidade | `temperature=0`; modelo, versão do prompt, data e versão do Python registrados em `RUN_INFO` e salvos junto dos resultados | arquivo `baseline_v1_resultados.json` |
| **RNF-05** | Rastreabilidade | para todo caso é possível reconstruir pergunta → código → resultado bruto → texto final | colunas `codigo`, `resultado_bruto` e `resposta` na tabela da seção 17 |
| **RNF-06** | Segurança da execução | o código gerado não pode importar módulos, acessar arquivos, rede ou atributos privados | lista de tokens proibidos aplicada **antes** de executar (`codigo_seguro`, seção 13) |
| **RNF-07** | Custo de preparação dos dados | o sistema **não** processa 1,8 GB em tempo de consulta; carrega um artefato de 0,6 MB | tempo de carga medido na seção 12 |

## Restrições

- **Modelo fixo pela disciplina** (`openai/gpt-oss-20b` ou `llama-3.3-70b-versatile` via Groq);
  não há *fine-tuning*.
- **Nenhuma chave de API escrita no notebook** — carregada de `userdata` (Colab) ou de
  variável de ambiente.
- `temperature=0` reduz variação, mas **não garante** saídas idênticas em serviço de
  inferência distribuída; conclusões sobre diferença entre versões exigem repetição.
- O artefato de dados é congelado junto com o conjunto de avaliação: mudar o ETL obriga a
  reexecutar o baseline.


# 8. Recursos externos potencialmente necessários

## Já em uso nesta versão

| Recurso | Papel | Situação |
|---|---|---|
| Groq API (`openai/gpt-oss-20b`) | planejamento da consulta (1 chamada) | disponível |
| `langchain` / `langchain-groq` | cliente e saída estruturada (`with_structured_output`) | disponível |
| `pydantic` | validação do esquema de saída (RNF-01) | disponível |
| `pandas` | execução da consulta sobre o artefato | disponível |
| INEP — Microdados do ENEM 2023 | fonte primária das notas | baixado uma vez no ETL (`etl/build_dataset.py`) |
| IBGE — API de Agregados v3 (6579 e 5938) | população estimada e PIB municipal de 2021 | baixado uma vez no ETL |
| IBGE — API de Localidades v1 | nome, UF e região de cada município | baixado uma vez no ETL |

## Hipóteses para as próximas versões

| Recurso | Para quê | Entregável previsto |
|---|---|---|
| `langgraph` | grafo Planejador → Supervisor → agentes especializados | 2 |
| `subprocess` com *timeout* e diretório restrito | sandbox real de execução do código gerado (substitui a lista de tokens) | 2 |
| Log estruturado em JSON (um registro por transição de agente) | observabilidade; base para discutir confiabilidade | 2 |
| `matplotlib` | agente Visualizador (RF-08) | 3 |
| LLM como juiz, calibrado contra os casos manuais | avaliar respostas abertas e ambíguas (RF-09) | 3–4 |
| Servidor MCP para a API do IBGE | consultar indicadores fora do artefato pré-processado, sem novo ETL | 4, se houver folga |
| Microdados do ENEM de outros anos | séries temporais | 4, apenas se a avaliação estiver estável |

> Nenhum destes é adotado agora. Cada um deve ser justificado, no entregável em que
> entrar, por uma limitação **observada** na seção 18.


# 9. Tipo de baseline escolhido

## Classificação: **parcial**

O baseline executa o núcleo da tarefa — pergunta em português → consulta correta sobre o
recorte → resposta em português com evidência — para perguntas de **valor único, ranking e
cruzamento ENEM × IBGE**, que são os casos de uso UC-01 a UC-04. Ele **não** executa a
tarefa completa descrita na proposta: não gera gráfico (RF-08), não valida o próprio
resultado, não pede esclarecimento diante de ambiguidade (RF-09) e não adapta as ressalvas
à pergunta (RF-07 é atendido por uma nota fixa).

## (i) Por que essa escolha é adequada

É a solução mais simples que ainda resolve o problema declarado: **uma chamada ao LLM**
para planejar a consulta, seguida de execução e formatação **determinísticas**. Um baseline
mais simples — LLM respondendo direto, sem tocar nos dados — seria um espantalho: ele
alucinaria números e daria 0% de acerto, tornando qualquer arquitetura posterior
"melhor" sem informação. Um baseline mais complexo — já com validador e *retry* — gastaria
agora o incremento do Entregável 2 e apagaria justamente o ganho que se quer medir.

O prompt é **honesto**: descreve o esquema completo, dá as regras de agregação, instrui a
respeitar filtros de robustez e a recusar perguntas fora do recorte. É o prompt que o grupo
defenderia como tentativa séria.

## (ii) O que foi deliberadamente simplificado ou excluído

| Simplificação | Consequência esperada |
|---|---|
| Nenhuma validação do resultado | erros de coluna, de filtro ou de agregação passam direto para o usuário |
| Nenhum *retry* | um código que falha ao executar produz resposta vazia, não uma segunda tentativa |
| Uma única chamada ao LLM | o modelo escreve o texto **antes** de ver o resultado; a frase final é um *template* com um marcador `{resultado}` |
| Nota de limitação fixa | a ressalva sobre anos diferentes aparece mesmo quando a pergunta não cruza ENEM e IBGE |
| Segurança por lista de tokens proibidos | barra o acidente óbvio, não é isolamento real |
| Sem memória | *"e no ano anterior?"* não tem como funcionar |

## (iii) Como permitirá a comparação futura

O conjunto de avaliação (seção 14) é **congelado** e os resultados são salvos em
`baseline_v1_resultados.json` com `RUN_INFO`. Nos Entregáveis 2, 3 e 4 os mesmos 13 casos
serão executados contra a arquitetura da vez e comparados nas mesmas cinco medidas:
taxa de aprovação, acerto por tipo de caso, latência mediana, chamadas ao LLM e custo.
A pergunta que a comparação responde é direta: **cada agente acrescentado paga o custo de
latência e de tokens que ele impõe?**


# 10. Critérios preliminares de sucesso

| Critério | Requisito | Como será medido | Meta nesta versão |
|---|---|---|---|
| **Correção numérica** | RF-01, RF-03 | verificação determinística: valor executado × resposta de referência, tolerância relativa de 1% | ≥ 8 de 11 casos automáticos |
| **Cobertura de lista** | RF-02 | fração dos itens de referência presentes na resposta final | 100% no caso de ranking (T03) |
| **Fidelidade ao resultado** | RF-04 | o valor mostrado no texto é o valor devolvido pela execução | 100% das respostas viáveis |
| **Abstenção correta** | RF-05 | `viavel=False`, sem código executado, com marcador de ausência no texto | 3 de 3 (T09, T10, T11) |
| **Ausência de falsa abstenção** | RF-05 | o sistema **não** se abstém em pergunta respondível | 0 falsas abstenções nos casos T01–T08 |
| **Evidência disponível** | RF-06 | `codigo` e `resultado` não vazios em toda resposta viável | 100% |
| **Erro de execução** | RNF-01, RNF-06 | fração de casos em que o código gerado levanta exceção ou é bloqueado | ≤ 1 de 13 |
| **Latência** | RNF-02 | mediana de `latencia_s` sobre os 13 casos | < 10 s |
| **Custo** | RNF-03 | (tokens de entrada × preço + tokens de saída × preço); preços em <https://groq.com/pricing> | < US$ 0,05 no conjunto |
| **Ambiguidade** | RF-09 | **rubrica manual**: a resposta distingue as leituras possíveis ou declara a que adotou? Notas registradas na seção 17 | qualitativo, sem meta |

## Sobre a escolha das formas de verificação

- **Verificação determinística** para tudo que é numérico ou lista fechada — barata,
  objetiva e imune a paráfrase, porque compara o *resultado executado*, não o texto.
- **Rubrica humana** apenas para os dois casos ambíguos (T12, T13), onde não existe
  resposta única certa; as notas ficam registradas no notebook.
- **LLM como juiz** não é usado nesta versão: com 11 casos automáticos verificáveis por
  comparação numérica, ele acrescentaria custo e uma fonte de erro sem necessidade.
  Passa a fazer sentido no Entregável 3, quando o Sintetizador produzir texto aberto.

> Cuidado deliberado: correção é medida contra o **resultado da execução**, não contra a
> presença de uma palavra no texto. A seção 15 mostra, sem chamar o LLM, uma resposta que
> uma verificação ingênua aprovaria e que estas verificações reprovam.


# 11. Configuração do ambiente

A célula de chave procura `GROQ_API_KEY`, nesta ordem: variável de ambiente → arquivo
`.secret` no diretório atual ou em até quatro níveis acima (fica em `eval/.secret`,
compartilhado por todos os entregáveis; ignorado pelo git; modelo em `eval/.secret.example`)
→ `userdata` do Colab (nome `INF0093-2026-2S`, o mesmo do material da disciplina) → teclado.
**Nenhuma chave é escrita no notebook** (exigência do enunciado, seção 4).

`RUN_INFO` registra modelo, temperatura, versão do prompt e data: sem esse registro os
números desta execução não podem ser comparados com os das próximas arquiteturas
(seção 3.2 do enunciado).


In [1]:
%pip install -q -U langchain langchain-groq pydantic pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, getpass, datetime, platform
from pathlib import Path

def localizar_secret(inicio: Path = Path.cwd(), niveis: int = 4) -> Path | None:
    """Procura `.secret` no diretório atual e nos `niveis` diretórios acima.

    O arquivo fica em `eval/.secret`, compartilhado por todos os entregáveis;
    os notebooks rodam em `eval/release_N/solution/`, dois níveis abaixo.
    """
    for pasta in [inicio, *inicio.parents[:niveis]]:
        if (pasta / ".secret").is_file():
            return pasta / ".secret"
    return None

def carregar_chave_groq() -> str:
    """Procura a chave, nesta ordem: variável de ambiente, arquivo .secret,
    userdata do Colab, teclado. Nunca a exibe."""
    if os.environ.get("GROQ_API_KEY"):
        return "variável de ambiente"

    # Aceita `GROQ_API_KEY=gsk_...` ou só a chave. Modelo em eval/.secret.example.
    arquivo = localizar_secret()
    if arquivo:
        for linha in arquivo.read_text(encoding="utf-8").splitlines():
            linha = linha.strip()
            if not linha or linha.startswith("#"):
                continue
            chave = (linha.split("=", 1)[1] if "=" in linha else linha).strip().strip('"').strip("'")
            if chave:                                # placeholder vazio: segue adiante
                os.environ["GROQ_API_KEY"] = chave
                return f"arquivo {arquivo.parent.name}/.secret"

    try:
        from google.colab import userdata          # noqa: F401
        os.environ["GROQ_API_KEY"] = userdata.get("INF0093-2026-2S")
        return "Colab userdata"
    except Exception:
        os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ_API_KEY: ")
        return "entrada manual"

origem = carregar_chave_groq()
assert os.environ.get("GROQ_API_KEY"), "Chave não configurada."
print("Chave carregada via:", origem)

Chave carregada via: arquivo .secret


In [3]:
from langchain_groq import ChatGroq

# MODEL_NAME = "llama-3.3-70b-versatile"   # Meta
MODEL_NAME = "openai/gpt-oss-20b"          # OpenAI

TEMPERATURE = 0
PROMPT_VERSAO = "v1"

llm = ChatGroq(model=MODEL_NAME, temperature=TEMPERATURE)

# Registro da execução: acompanha os resultados até o Entregável 4.
#
RUN_INFO = {
    "modelo": MODEL_NAME,
    "temperatura": TEMPERATURE,
    "prompt_versao": PROMPT_VERSAO,
    "data": datetime.datetime.now().isoformat(timespec="seconds"),
    "python": platform.python_version(),
}
RUN_INFO

{'modelo': 'openai/gpt-oss-20b',
 'temperatura': 0,
 'prompt_versao': 'v1',
 'data': '2026-09-03T21:28:48',
 'python': '3.12.3'}

# 12. Dados

## O artefato e por que ele existe

O sistema **não** lê os microdados do ENEM em tempo de consulta. Um ETL executado uma única
vez, fora do sistema (`etl/build_dataset.py`), baixa os 550 MB do INEP, percorre o CSV de
1,8 GB em blocos, agrega por município da escola e cruza com duas APIs do IBGE. O resultado
é um CSV de **0,6 MB** com 5.481 linhas — pequeno o bastante para caber em memória e ter o
esquema inteiro descrito no prompt (RNF-07).

Essa separação é decisão de arquitetura, não conveniência: mantém o custo de cada consulta
independente do tamanho da base bruta e evita que o grupo gaste a primeira semana brigando
com ETL em vez de com a arquitetura de agentes.

## Critério de inclusão do candidato

Entram na agregação apenas candidatos que, simultaneamente: **(1)** declararam escola
(`CO_MUNICIPIO_ESC` preenchido), **(2)** estiveram presentes nos dois dias e **(3)** têm as
cinco notas. São **721.429 participantes** dos ~3,9 milhões de inscritos no ENEM 2023.

## Limitações do dado — registradas antes de qualquer resultado

1. **Recorte de candidatos.** O subconjunto com escola declarada é majoritariamente de
   concluintes do ensino médio regular. As médias **não** são a média do município como um todo.
2. **Município da escola ≠ município de residência ≠ município de prova.** A escolha é
   deliberada (é o vínculo com sentido educacional), mas muda o número.
3. **Anos diferentes.** ENEM de 2023, IBGE de 2021. Comparações socioeconômicas são aproximadas.
4. **Municípios pequenos.** O mínimo é 1 participante; ranking sem filtro de `n_participantes`
   é dominado por ruído. Perguntas de ranking no conjunto de avaliação trazem o filtro explícito.
5. **PIB per capita não é renda das famílias.** É produto por habitante — municípios com uma
   grande planta industrial aparecem no topo sem que isso reflita a renda local.
6. **Cobertura do cruzamento.** Os 5.481 códigos de município do INEP casaram 1:1 com o IBGE;
   o risco levantado na proposta do projeto **não** se materializou nesta edição.

O dicionário completo está em `dados/DICIONARIO.md`.


In [4]:
import time
import pandas as pd
from pathlib import Path

ARQUIVO = "enem2023_ibge_municipios.csv"
URL_DADOS = ("https://raw.githubusercontent.com/werner-denzin/unicamp.agents/main/"
             "04_projeto_sistema_multiagentes/eval/release_1/solution/dados/" + ARQUIVO)

def carregar_dados():
    """Procura o CSV ao lado do notebook, em ./dados e, por último, no repositório."""
    for origem in (Path("dados") / ARQUIVO, Path(ARQUIVO), URL_DADOS):
        try:
            return pd.read_csv(origem), str(origem)
        except Exception:
            continue
    raise FileNotFoundError(
        f"{ARQUIVO} não encontrado. Gere-o com etl/build_dataset.py "
        f"ou coloque-o ao lado do notebook."
    )

inicio = time.perf_counter()
df, origem_dados = carregar_dados()
TEMPO_CARGA = time.perf_counter() - inicio

print(f"origem   : {origem_dados}")
print(f"formato  : {df.shape[0]} municípios × {df.shape[1]} colunas")
print(f"carga    : {TEMPO_CARGA:.2f} s   (RNF-07)")
print(f"nulos    : {int(df.isna().sum().sum())}")
print(f"cobertura: {int(df['n_participantes'].sum()):,} participantes".replace(",", "."))
df.head(3)

origem   : dados/enem2023_ibge_municipios.csv
formato  : 5481 municípios × 15 colunas
carga    : 0.01 s   (RNF-07)
nulos    : 0
cobertura: 721.429 participantes


,co_municipio,municipio,uf,regiao,populacao_2021,pib_2021_mil_reais,n_participantes,media_cn,media_ch,media_lc,media_mt,media_redacao,media_geral,pct_escola_publica,pib_per_capita_2021
0,1200013,Acrelândia,AC,Norte,15721.0,398725.0,34,476.39,486.03,493.29,486.84,560.00,500.51,100.0,25362.57
1,1200054,Assis Brasil,AC,Norte,7649.0,133916.0,8,462.34,500.61,494.55,455.30,437.50,470.06,100.0,17507.65
2,1200104,Brasiléia,AC,Norte,27123.0,685636.0,65,458.61,484.94,486.28,469.83,531.38,486.21,100.0,25278.77


## Esquema entregue ao modelo

O modelo nunca vê os dados — vê apenas a descrição abaixo. Ela é gerada a partir do próprio
`DataFrame`, para não sair de sincronia com o arquivo, e é o único contexto que o baseline
tem para decidir **se** a pergunta é respondível e **como** respondê-la.


In [5]:
DESCRICOES = {
    "co_municipio":        "código IBGE do município (7 dígitos)",
    "municipio":           "nome do município (ex.: 'Campinas', 'São Paulo')",
    "uf":                  "sigla da UF (ex.: 'SP', 'BA')",
    "regiao":              "Norte, Nordeste, Centro-Oeste, Sudeste ou Sul",
    "populacao_2021":      "população residente estimada em 2021 (IBGE)",
    "pib_2021_mil_reais":  "PIB municipal de 2021 em MIL reais (IBGE)",
    "n_participantes":     "candidatos do ENEM 2023 considerados na média do município",
    "media_cn":            "média da nota de Ciências da Natureza (0 a 1000)",
    "media_ch":            "média da nota de Ciências Humanas (0 a 1000)",
    "media_lc":            "média da nota de Linguagens e Códigos (0 a 1000)",
    "media_mt":            "média da nota de Matemática (0 a 1000)",
    "media_redacao":       "média da nota de Redação (0 a 1000)",
    "media_geral":         "média das cinco notas por candidato, agregada por município",
    "pct_escola_publica":  "percentual de participantes de escola pública (0 a 100)",
    "pib_per_capita_2021": "PIB por habitante em 2021, em reais",
}

def descrever_esquema(df: pd.DataFrame) -> str:
    linhas = [f"- {c} ({df[c].dtype}): {DESCRICOES[c]}" for c in df.columns]
    return "\n".join(linhas)

ESQUEMA = descrever_esquema(df)
print(ESQUEMA)

- co_municipio (int64): código IBGE do município (7 dígitos)
- municipio (str): nome do município (ex.: 'Campinas', 'São Paulo')
- uf (str): sigla da UF (ex.: 'SP', 'BA')
- regiao (str): Norte, Nordeste, Centro-Oeste, Sudeste ou Sul
- populacao_2021 (float64): população residente estimada em 2021 (IBGE)
- pib_2021_mil_reais (float64): PIB municipal de 2021 em MIL reais (IBGE)
- n_participantes (int64): candidatos do ENEM 2023 considerados na média do município
- media_cn (float64): média da nota de Ciências da Natureza (0 a 1000)
- media_ch (float64): média da nota de Ciências Humanas (0 a 1000)
- media_lc (float64): média da nota de Linguagens e Códigos (0 a 1000)
- media_mt (float64): média da nota de Matemática (0 a 1000)
- media_redacao (float64): média da nota de Redação (0 a 1000)
- media_geral (float64): média das cinco notas por candidato, agregada por município
- pct_escola_publica (float64): percentual de participantes de escola pública (0 a 100)
- pib_per_capita_2021 (float6

# 13. Implementação do baseline

## Desenho

```
pergunta ──► [LLM, 1 chamada] ──► plano estruturado ──► [guarda de segurança] ──► [eval pandas] ──► [formatação] ──► resposta
                                   viavel?                tokens proibidos          determinístico    determinística
                                   codigo_pandas
                                   template_resposta
```

A **única** chamada ao LLM produz um plano: se a pergunta é respondível, qual expressão
pandas responde e qual frase apresenta o resultado. Tudo depois disso é determinístico.

Duas consequências desse desenho, ambas deliberadas:

- o modelo escreve o texto **antes** de ver o número, então a frase é um *template* com o
  marcador `{resultado}` — é exatamente essa limitação que o agente Sintetizador do
  Entregável 3 deve resolver;
- não há validação nem *retry* — um código errado produz um número errado, visível na
  seção 17. É o que torna o ganho do Validador (Entregável 2) mensurável.

## Segurança da execução (RNF-06)

O código gerado passa antes por uma guarda sintática: `ast.parse(mode="eval")` garante que é
**uma única expressão** (atribuição, `import` e ponto e vírgula viram erro de sintaxe) e um
passeio pela árvore recusa qualquer nome fora de `df`, `pd`, `np` e uma lista curta de
builtins, além de atributos privados. A execução usa `eval` com `__builtins__` substituído.

**Isto não é um sandbox.** Roda no mesmo processo, sem *timeout* e sem limite de memória: um
laço infinito derruba o notebook. O isolamento real (subprocesso, *timeout*, sem rede, sem
filesystem) é tarefa do Entregável 2 e está registrado como limitação na seção 18.


In [6]:
from typing import Any, Optional
from pydantic import BaseModel, Field

class PlanoConsulta(BaseModel):
    """O que a única chamada ao LLM precisa devolver."""
    viavel: bool = Field(
        description="True se a pergunta pode ser respondida SOMENTE com as colunas listadas.")
    motivo: str = Field(
        description="Se viavel=False, qual informação falta. Se viavel=True, a estratégia em uma frase.")
    codigo_pandas: str = Field(
        description="UMA expressão pandas sobre o DataFrame `df`. String vazia se viavel=False.")
    template_resposta: str = Field(
        description="Frase em português contendo o marcador {resultado}. Vazia se viavel=False.")
    colunas_usadas: list[str] = Field(
        description="Colunas do esquema usadas na expressão.")

# method="json_schema" usa a saída estruturada nativa da Groq. O padrão (tool calling)
# falhou na primeira execução real: com o prompt completo, o gpt-oss-20b tentou chamar uma
# ferramenta chamada "json" em vez de PlanoConsulta e a API devolveu 400
# ("tool call validation failed"). Registrado na seção 18.
structured_llm = llm.with_structured_output(PlanoConsulta, method="json_schema", include_raw=True)
print("[done]")

[done]


In [7]:
INSTRUCAO = """
Você traduz perguntas em português para consultas pandas sobre um DataFrame chamado `df`,
já carregado, com uma linha por município brasileiro.

COLUNAS DISPONÍVEIS (são as ÚNICAS existentes):
{esquema}

O RECORTE DOS DADOS:
- ENEM de 2023 apenas; indicadores do IBGE de 2021 apenas.
- Só entram candidatos que declararam escola, estiveram presentes nos dois dias e têm as
  cinco notas: 721.429 participantes em 5.481 municípios.
- O vínculo municipal é o município DA ESCOLA, não o de residência nem o de prova.

REGRAS:
1. Se a pergunta exigir qualquer informação que não esteja nas colunas acima — outro ano,
   outro exame, outro indicador (IDH, renda familiar, número de escolas), nota por
   disciplina específica (inglês, física, química), dado por candidato — responda com
   viavel=false, codigo_pandas="" e explique em `motivo` exatamente o que falta.
   NUNCA invente um número e nunca aproxime usando uma coluna diferente da pedida.
2. Se a pergunta for respondível, `codigo_pandas` deve ser UMA ÚNICA EXPRESSÃO Python
   (sem `import`, sem atribuição, sem `print`, sem ponto e vírgula) avaliada sobre `df`.
   Os únicos nomes disponíveis são `df`, `pd`, `np` e as funções round, len, sorted, list,
   min, max, sum, abs, float, int, str. Qualquer outro nome faz a execução ser recusada.
3. Municípios homônimos existem: filtre também por `uf` quando a pergunta citar o estado.
   "São Paulo" como município é `(df.municipio == "São Paulo") & (df.uf == "SP")`;
   "estado de São Paulo" é `df.uf == "SP"`.
4. Respeite filtros de robustez pedidos na pergunta (ex.: "com pelo menos 100
   participantes" vira `df.n_participantes >= 100`). Não invente filtros que não foram pedidos.
5. Para valor único devolva um escalar; para ranking devolva um DataFrame com as colunas
   relevantes já ordenado e limitado (`.nlargest`, `.head`); para comparação entre grupos
   devolva uma Series indexada pelo grupo.
6. `template_resposta` é uma frase em português que apresenta o resultado e contém
   EXATAMENTE UMA VEZ o marcador {{resultado}}. Não escreva o número você mesmo — você
   ainda não o conhece. Não use chaves para mais nada.

EXEMPLOS (formato, não conteúdo):

Pergunta: "Quantos municípios da Bahia estão na base?"
  viavel=true
  codigo_pandas: int((df.uf == "BA").sum())
  template_resposta: "A base tem {{resultado}} municípios da Bahia."

Pergunta: "Qual o número de professores por aluno em Natal?"
  viavel=false
  motivo: "O recorte não tem dados de docentes; as colunas cobrem notas do ENEM 2023,
           população e PIB municipais."
  codigo_pandas: ""
"""

def montar_prompt(pergunta: str) -> str:
    return INSTRUCAO.format(esquema=ESQUEMA) + f"\n\nPERGUNTA:\n{pergunta}\n"

print(montar_prompt("Qual é a média geral em Campinas?")[:400], "...")


Você traduz perguntas em português para consultas pandas sobre um DataFrame chamado `df`,
já carregado, com uma linha por município brasileiro.

COLUNAS DISPONÍVEIS (são as ÚNICAS existentes):
- co_municipio (int64): código IBGE do município (7 dígitos)
- municipio (str): nome do município (ex.: 'Campinas', 'São Paulo')
- uf (str): sigla da UF (ex.: 'SP', 'BA')
- regiao (str): Norte, Nordeste, Ce ...


In [8]:
import ast
import numpy as np

# RNF-06 — guarda sintática. NÃO é um sandbox: ver seção 18.
#
NOMES_PERMITIDOS = {"df", "pd", "np"}
BUILTINS_PERMITIDOS = {
    "round": round, "len": len, "sorted": sorted, "list": list, "min": min,
    "max": max, "sum": sum, "abs": abs, "float": float, "int": int, "str": str,
    "dict": dict, "set": set, "zip": zip, "range": range, "bool": bool,
    "enumerate": enumerate, "True": True, "False": False, "None": None,
}

def codigo_seguro(codigo: str) -> list[str]:
    """Devolve os motivos de recusa. Lista vazia = liberado.

    `ast.parse(mode="eval")` já garante UMA expressão: atribuição, `import`,
    `print` e ponto e vírgula viram erro de sintaxe. Sobra checar quais nomes a
    expressão toca, daí o passeio pela árvore.
    """
    try:
        arvore = ast.parse(codigo, mode="eval")
    except SyntaxError as e:
        return [f"não é uma expressão única: {e.msg}"]

    permitidos = NOMES_PERMITIDOS | set(BUILTINS_PERMITIDOS)
    motivos = []
    for no in ast.walk(arvore):
        if isinstance(no, ast.Name) and no.id not in permitidos:
            motivos.append(f"nome não permitido: {no.id}")
        if isinstance(no, ast.Attribute) and no.attr.startswith("_"):
            motivos.append(f"atributo privado: {no.attr}")
    return motivos

def executar(codigo: str):
    """Executa a expressão em espaço de nomes restrito. Devolve (valor, erro)."""
    motivos = codigo_seguro(codigo)
    if motivos:
        return None, f"código bloqueado pela guarda de segurança: {motivos}"
    try:
        return eval(codigo, {"__builtins__": BUILTINS_PERMITIDOS},
                    {"df": df, "pd": pd, "np": np}), None
    except Exception as e:
        return None, f"{type(e).__name__}: {e}"

print("[done]")

[done]


## A guarda em ação

Demonstração sem LLM: expressões legítimas passam, as demais são recusadas **antes** de
qualquer execução.


In [9]:
# RNF-06 — a guarda recusa antes de executar. Nenhuma destas expressões roda.
for tentativa in [
    'df.media_geral.mean()',                       # legítima
    'df.sort_values("media_mt", ascending=False).head(3)["municipio"]',   # kwargs: permitido
    '__import__("os").system("ls")',               # nome não permitido
    'df.to_csv("vazou.csv")',                      # nome permitido, mas escreve em disco
    'x = df.media_geral.mean()',                   # não é expressão única
    'df.__class__.__mro__',                        # atributo privado
]:
    motivos = codigo_seguro(tentativa)
    print(f'{"LIBERA" if not motivos else "RECUSA"}  {tentativa}')
    if motivos:
        print(f'         {motivos}')

LIBERA  df.media_geral.mean()
LIBERA  df.sort_values("media_mt", ascending=False).head(3)["municipio"]
RECUSA  __import__("os").system("ls")
         ['nome não permitido: __import__']
LIBERA  df.to_csv("vazou.csv")
RECUSA  x = df.media_geral.mean()
         ['não é uma expressão única: invalid syntax']
RECUSA  df.__class__.__mro__
         ['atributo privado: __mro__', 'atributo privado: __class__']


In [10]:
NOTA_RECORTE = (
    "Recorte: ENEM 2023, apenas candidatos com escola declarada, presentes nos dois dias "
    "e com as cinco notas (721.429 de ~3,9 milhões de inscritos), agregados pelo município "
    "da escola. Indicadores do IBGE são de 2021."
)

def formatar_valor(valor) -> str:
    """Converte o resultado bruto da execução em texto legível (determinístico)."""
    if valor is None:
        return "sem resultado"
    if isinstance(valor, (bool, np.bool_)):
        return "sim" if valor else "não"
    if isinstance(valor, (int, np.integer)):
        return str(int(valor))
    if isinstance(valor, (float, np.floating)):
        # correlações e proporções perdem sentido com duas casas
        casas = 4 if abs(float(valor)) < 1 else 2
        texto = f"{float(valor):.{casas}f}"
        return texto.rstrip("0").rstrip(".") if "." in texto else texto
    if isinstance(valor, pd.Series):
        return "; ".join(f"{i}: {formatar_valor(v)}" for i, v in valor.head(10).items())
    if isinstance(valor, pd.DataFrame):
        rotulos = [c for c in valor.columns if valor[c].dtype == object or
                   pd.api.types.is_string_dtype(valor[c])]
        numeros = [c for c in valor.columns if c not in rotulos and c != "co_municipio"]
        linhas = []
        for _, linha in valor.head(10).iterrows():
            nome = " / ".join(str(linha[c]) for c in rotulos)
            medidas = ", ".join(f"{c}: {formatar_valor(linha[c])}" for c in numeros)
            linhas.append(f"{nome} ({medidas})" if nome and medidas else (nome or medidas))
        return "; ".join(linhas)
    return str(valor)

class Resposta(BaseModel):
    texto: str
    resultado: Any = None
    codigo: str = ""
    viavel: bool = True
    motivo: str = ""
    erro_execucao: Optional[str] = None

def baseline(pergunta: str) -> tuple[Resposta, dict]:
    """Baseline: UMA chamada ao LLM + execução e formatação determinísticas."""
    inicio = time.perf_counter()
    try:
        saida = structured_llm.invoke(montar_prompt(pergunta))
    except Exception as e:                              # erro de API conta como falha, não derruba o notebook
        latencia = time.perf_counter() - inicio
        erro = f"erro na chamada ao modelo: {type(e).__name__}: {str(e)[:200]}"
        return Resposta(texto=f"Falha na chamada ao modelo. {erro}", erro_execucao=erro), {
            "latencia_s": round(latencia, 2), "tokens_entrada": None, "tokens_saida": None,
            "chamadas_llm": 1, "erro_parse": erro}
    latencia = time.perf_counter() - inicio

    uso = getattr(saida["raw"], "usage_metadata", None) or {}
    metricas = {
        "latencia_s": round(latencia, 2),
        "tokens_entrada": uso.get("input_tokens"),
        "tokens_saida": uso.get("output_tokens"),
        "chamadas_llm": 1,
        "erro_parse": str(saida["parsing_error"]) if saida["parsing_error"] else None,
    }

    plano = saida["parsed"]
    if plano is None:                                   # RNF-01 violado
        return Resposta(texto="Falha ao interpretar a saída do modelo.",
                        viavel=False, motivo=str(metricas["erro_parse"])), metricas

    if not plano.viavel:                                # RF-05
        return Resposta(texto=f"Não é possível responder com os dados disponíveis. {plano.motivo}",
                        viavel=False, motivo=plano.motivo), metricas

    valor, erro = executar(plano.codigo_pandas)
    if erro:
        return Resposta(texto=f"A consulta gerada não pôde ser executada. {erro}",
                        codigo=plano.codigo_pandas, motivo=plano.motivo,
                        erro_execucao=erro), metricas

    texto_valor = formatar_valor(valor)
    try:
        frase = plano.template_resposta.format(resultado=texto_valor)
    except (KeyError, IndexError, ValueError):          # template malformado
        frase = f"{plano.template_resposta} {texto_valor}".strip()

    return Resposta(texto=f"{frase}\n\n{NOTA_RECORTE}", resultado=valor,
                    codigo=plano.codigo_pandas, motivo=plano.motivo), metricas

print("[done]")

[done]


## Primeira execução — uma pergunta viável e uma inviável

As duas perguntas abaixo **não** fazem parte do conjunto de avaliação: servem só para
mostrar o comportamento antes de congelar a régua.


In [11]:
for pergunta in ["Qual é a média de redação no estado do Ceará?",
                 "Quantas escolas particulares existem em Fortaleza?"]:
    r, m = baseline(pergunta)
    print("=" * 78)
    print("PERGUNTA :", pergunta)
    print("VIÁVEL   :", r.viavel)
    print("CÓDIGO   :", r.codigo or "—")
    print("RESPOSTA :", r.texto.split("\n")[0])
    print("MÉTRICAS :", m)

PERGUNTA : Qual é a média de redação no estado do Ceará?
VIÁVEL   : True
CÓDIGO   : df.loc[df.uf == 'CE', 'media_redacao'].mean()
RESPOSTA : A média de redação no estado do Ceará é 531.47.
MÉTRICAS : {'latencia_s': 0.69, 'tokens_entrada': 1327, 'tokens_saida': 379, 'chamadas_llm': 1, 'erro_parse': None}


PERGUNTA : Quantas escolas particulares existem em Fortaleza?
VIÁVEL   : False
CÓDIGO   : —
RESPOSTA : Não é possível responder com os dados disponíveis. Não há coluna que indique o número de escolas particulares; a base contém apenas percentuais de participantes de escolas públicas e dados de ENEM e PIB.
MÉTRICAS : {'latencia_s': 0.59, 'tokens_entrada': 1323, 'tokens_saida': 344, 'chamadas_llm': 1, 'erro_parse': None}


# 14. Conjunto de avaliação

**13 casos**, dos quais 11 verificados automaticamente e 2 por rubrica manual. As respostas
de referência foram calculadas com uma consulta pandas escrita à mão sobre o mesmo artefato
— **antes** de qualquer chamada ao LLM e antes de qualquer ajuste no prompt.

Este conjunto está **congelado**: os mesmos 13 casos serão executados nos Entregáveis 2, 3 e
4. Se ele mudar, o baseline precisa ser reexecutado, porque comparação exige a mesma régua.

| ID | Pergunta | Tipo | Referência | Verificação |
|---|---|---|---|---|
| T01 | Média geral em Campinas (SP) | normal / valor único | 584,24 | auto (numérica, 1%) |
| T02 | Quantos municípios do Acre há na base | normal / contagem | 22 | auto (numérica, exata) |
| T03 | Top 5 de SP em matemática, com ≥ 100 participantes | ranking com filtro | Valinhos, São João da Boa Vista, Amparo, São José dos Campos, Jaú | auto (cobertura de lista) |
| T04 | Região com maior média de redação | agregação por grupo | Sudeste | auto (nome na resposta) |
| T05 | Quantos municípios têm média geral acima de 550 | filtro + contagem | 990 | auto (numérica, exata) |
| T06 | Correlação PIB per capita × média geral (≥ 50 participantes) | **cruzamento ENEM×IBGE** | 0,2868 | auto (numérica, 5%) |
| T07 | Entre os 10 mais populosos, qual tem maior média de matemática | **cruzamento + composto** | Belo Horizonte (638,58) | auto (nome + valor) |
| T08 | Média geral do Nordeste × média geral do Sul | comparação entre grupos | 488,64 e 528,01 | auto (dois valores) |
| T09 | Nota média de **inglês** em Salvador | informação ausente | abstenção | auto (abstenção) |
| T10 | **IDH** de Recife | informação ausente | abstenção | auto (abstenção) |
| T11 | Média geral do ENEM **2019** em Belo Horizonte | fora do recorte temporal | abstenção | auto (abstenção) |
| T12 | "Qual é o melhor município para estudar?" | ambíguo | — | **manual** |
| T13 | "Qual é a média de São Paulo?" | ambíguo (capital × estado; qual média?) | — | **manual** |

## Como os tipos foram escolhidos

- **T01–T05** cobrem os padrões de consulta mais frequentes: valor único, contagem,
  ranking com filtro de robustez e agregação por grupo.
- **T06–T08** são o diferencial do projeto: exigem o cruzamento ENEM × IBGE ou uma
  composição de duas operações (filtrar os 10 maiores, depois maximizar outra coluna).
- **T09–T11** são as armadilhas. Cada uma falha por um motivo diferente: coluna que não
  existe (inglês), indicador que não existe (IDH) e **ano que não existe** (2019) — este
  último é o mais perigoso, porque o modelo tem os dados de 2019 na memória paramétrica e
  pode responder sem tocar no `DataFrame`.
- **T12–T13** não têm resposta única. Não são "casos perdidos": servem para medir, ao longo
  dos entregáveis, se o sistema passa a pedir esclarecimento (RF-09) em vez de escolher uma
  leitura em silêncio.

> Com 11 casos automáticos, um acerto vale 9 pontos percentuais. É pouco para afirmar
> superioridade estatística entre arquiteturas; o conjunto será ampliado para 25–30 casos
> no Entregável 4, como previsto no planejamento do projeto.


In [12]:
# CONGELADO em 07/09/2026. As referências foram calculadas com consultas pandas
# escritas à mão sobre este mesmo artefato, antes de qualquer chamada ao LLM.
#
test_cases = [
    {"id": "T01", "tipo": "normal", "verificacao": "auto", "criterio": "numerico",
     "pergunta": "Qual é a média geral do ENEM 2023 no município de Campinas, em São Paulo?",
     "esperado": 584.24, "tolerancia": 0.01,
     "referencia": 'df.loc[(df.municipio == "Campinas") & (df.uf == "SP"), "media_geral"].item()'},

    {"id": "T02", "tipo": "normal", "verificacao": "auto", "criterio": "numerico",
     "pergunta": "Quantos municípios do estado do Acre estão na base?",
     "esperado": 22, "tolerancia": 0.0,
     "referencia": 'int((df.uf == "AC").sum())'},

    {"id": "T03", "tipo": "ranking", "verificacao": "auto", "criterio": "lista",
     "pergunta": ("Quais são os 5 municípios de São Paulo com maior média em matemática, "
                  "considerando apenas municípios com pelo menos 100 participantes?"),
     "esperado": ["Valinhos", "São João da Boa Vista", "Amparo",
                  "São José dos Campos", "Jaú"],
     "cobertura_minima": 1.0,
     "referencia": 'df[(df.uf == "SP") & (df.n_participantes >= 100)].nlargest(5, "media_mt")'},

    {"id": "T04", "tipo": "agregação por grupo", "verificacao": "auto", "criterio": "lista",
     "pergunta": "Qual região do país tem a maior média de redação?",
     "esperado": ["Sudeste"], "cobertura_minima": 1.0,
     "referencia": 'df.groupby("regiao")["media_redacao"].mean().idxmax()'},

    {"id": "T05", "tipo": "normal", "verificacao": "auto", "criterio": "numerico",
     "pergunta": "Quantos municípios têm média geral acima de 550?",
     "esperado": 990, "tolerancia": 0.0,
     "referencia": 'int((df.media_geral > 550).sum())'},

    {"id": "T06", "tipo": "cruzamento ENEM×IBGE", "verificacao": "auto", "criterio": "numerico",
     "pergunta": ("Qual é a correlação entre o PIB per capita de 2021 e a média geral do ENEM "
                  "dos municípios com pelo menos 50 participantes?"),
     "esperado": 0.2868, "tolerancia": 0.05,
     "referencia": ('df[df.n_participantes >= 50]["pib_per_capita_2021"]'
                    '.corr(df[df.n_participantes >= 50]["media_geral"])')},

    {"id": "T07", "tipo": "cruzamento + composto", "verificacao": "auto", "criterio": "misto",
     "pergunta": ("Entre os 10 municípios mais populosos do país, qual tem a maior média "
                  "em matemática?"),
     "esperado": 638.58, "tolerancia": 0.01, "esperado_texto": ["Belo Horizonte"],
     "referencia": 'df.nlargest(10, "populacao_2021").nlargest(1, "media_mt")'},

    {"id": "T08", "tipo": "comparação entre grupos", "verificacao": "auto", "criterio": "lista",
     "pergunta": ("Compare a média geral dos municípios do Nordeste com a dos municípios "
                  "do Sul."),
     "esperado": ["488", "528"], "cobertura_minima": 1.0,
     "referencia": 'df.groupby("regiao")["media_geral"].mean()[["Nordeste", "Sul"]]'},

    {"id": "T09", "tipo": "informação ausente", "verificacao": "auto", "criterio": "abstencao",
     "pergunta": "Qual é a nota média de inglês no ENEM 2023 em Salvador?",
     "esperado": None,
     "referencia": "não existe nota por língua estrangeira no artefato"},

    {"id": "T10", "tipo": "informação ausente", "verificacao": "auto", "criterio": "abstencao",
     "pergunta": "Qual é o IDH de Recife?",
     "esperado": None,
     "referencia": "IDH não é indicador do artefato (só população, PIB e PIB per capita)"},

    {"id": "T11", "tipo": "fora do recorte temporal", "verificacao": "auto", "criterio": "abstencao",
     "pergunta": "Qual foi a média geral do ENEM 2019 em Belo Horizonte?",
     "esperado": None,
     "referencia": "o artefato só tem 2023; o risco é o modelo responder de memória"},

    {"id": "T12", "tipo": "ambíguo", "verificacao": "manual", "criterio": "manual",
     "pergunta": "Qual é o melhor município para estudar?",
     "esperado": None,
     "nota": ("Boa resposta: apontar que 'melhor' não está definido, oferecer um critério "
              "(ex.: maior media_geral com n_participantes mínimo) e declarar a escolha. "
              "Resposta ruim: devolver um município sem dizer sob qual critério.")},

    {"id": "T13", "tipo": "ambíguo", "verificacao": "manual", "criterio": "manual",
     "pergunta": "Qual é a média de São Paulo?",
     "esperado": None,
     "nota": ("Duplamente ambíguo: capital ou estado? média de qual área? Boa resposta: "
              "pedir esclarecimento ou responder declarando explicitamente a leitura "
              "adotada (ex.: 'município de São Paulo, média geral').")},
]

print(len(test_cases), "casos;",
      sum(c["verificacao"] == "auto" for c in test_cases), "automáticos;",
      sum(c["verificacao"] == "manual" for c in test_cases), "manuais.")

13 casos; 11 automáticos; 2 manuais.


## Sanidade das referências

A célula abaixo reexecuta as consultas de referência sobre o artefato carregado. Ela **não**
usa o LLM: existe para garantir que os valores da tabela acima continuam válidos para este
arquivo de dados. Se alguma linha divergir, o artefato mudou e o baseline precisa ser
reexecutado.


In [13]:
conferencia = []
for caso in test_cases:
    if caso["criterio"] in ("abstencao", "manual"):
        continue
    valor, erro = executar(caso["referencia"])
    conferencia.append({"id": caso["id"], "referencia_executada": formatar_valor(valor),
                        "erro": erro})

pd.DataFrame(conferencia)

,id,referencia_executada,erro
0,T01,584.24,None
1,T02,22,None
2,T03,Valinhos / SP / Sudeste (populacao_2021: 13316...,None
3,T04,Sudeste,None
4,T05,990,None
5,T06,0.2868,None
6,T07,Belo Horizonte / MG / Sudeste (populacao_2021:...,None
7,T08,Nordeste: 488.64; Sul: 528.01,None


# 15. Implementação da verificação

Cinco verificações, cada uma ligada a um requisito da seção 6:

| Função | Requisito | O que mede |
|---|---|---|
| `resultado_correto` | RF-01, RF-03 | o **valor executado** bate com a referência, dentro da tolerância |
| `cobertura_lista` | RF-02 | fração dos itens de referência presentes na resposta final |
| `texto_fiel` | RF-04 | o número mostrado ao usuário é o número que a execução devolveu |
| `abstencao_valida` | RF-05 | recusou, não executou código e disse o que falta |
| `tem_evidencia` | RF-06 | há código e resultado bruto disponíveis para auditoria |

O ponto central: **correção é medida sobre o resultado da execução**, não sobre a presença de
uma palavra no texto. Isso torna a verificação imune a paráfrase e é possível justamente
porque o baseline expõe o valor bruto.


In [14]:
import re, unicodedata

def normalizar(texto: str) -> str:
    texto = unicodedata.normalize("NFKD", str(texto).lower())
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", texto).strip()

# Números aparecem em três formatos no notebook: pt-BR ("1.234,56"), inglês
# ("1,234.56") e simples ("584.24"). Confundi-los reprova resposta certa.
#
# Armadilha: "0.286" é milhar pt-BR ou decimal? `formatar_valor` emite decimal com
# ponto, então um ponto só é separador de milhar quando há DOIS ou mais grupos
# ("1.234.567") ou quando uma vírgula decimal vem depois ("1.234,56").
PT_BR_MILHAR = r"-?\d{1,3}(?:\.\d{3}){2,}(?:,\d+)?|-?\d{1,3}(?:\.\d{3})+,\d+"
EN_MILHAR = r"-?\d{1,3}(?:,\d{3})+(?:\.\d+)?"
PADRAO_NUMERO = re.compile(
    PT_BR_MILHAR                             # 1.234.567 / 1.234,56  (pt-BR)
    + "|" + EN_MILHAR                        # 1,234,567.89          (inglês)
    + r"|-?\d+(?:[.,]\d+)?"                 # 584.24, 584,24, 0.2868
)

def _para_float(texto: str) -> float:
    if re.fullmatch(PT_BR_MILHAR, texto):
        return float(texto.replace(".", "").replace(",", "."))
    if re.fullmatch(EN_MILHAR, texto):
        return float(texto.replace(",", ""))
    return float(texto.replace(",", "."))

def numeros_de(valor) -> list[float]:
    """Todos os números contidos em um escalar, Series, DataFrame ou texto."""
    if isinstance(valor, (bool, np.bool_)):
        return []
    if isinstance(valor, (int, float, np.integer, np.floating)):
        return [float(valor)]
    if isinstance(valor, pd.Series):
        return [float(v) for v in pd.to_numeric(valor, errors="coerce").dropna()]
    if isinstance(valor, pd.DataFrame):
        numericas = valor.select_dtypes("number")
        return [float(v) for v in numericas.to_numpy().ravel() if pd.notna(v)]
    return [_para_float(t) for t in PADRAO_NUMERO.findall(str(valor))]

def proximo(obtido: float, esperado: float, tolerancia: float) -> bool:
    return abs(obtido - esperado) <= tolerancia * max(abs(esperado), 1.0)

def resultado_correto(caso: dict, resposta) -> bool:
    """RF-01/RF-03: o valor executado contém a referência dentro da tolerância."""
    if resposta.resultado is None:
        return False
    return any(proximo(n, caso["esperado"], caso.get("tolerancia", 0.01))
               for n in numeros_de(resposta.resultado))

def cobertura_lista(caso: dict, resposta) -> float:
    """RF-02: fração dos itens de referência presentes no texto final."""
    texto = normalizar(resposta.texto)
    itens = caso["esperado"]
    return sum(normalizar(i) in texto for i in itens) / len(itens)

def texto_fiel(resposta) -> bool:
    """RF-04: o número mostrado ao usuário é um número devolvido pela execução."""
    if resposta.resultado is None:
        return False
    do_resultado = numeros_de(resposta.resultado)
    if not do_resultado:
        return True                      # resultado não numérico (ex.: nome de região)
    do_texto = numeros_de(resposta.texto.split("Recorte:")[0])
    return any(proximo(t, r, 0.011) for r in do_resultado for t in do_texto)

MARCADORES_AUSENCIA = [
    "nao e possivel", "nao esta", "nao consta", "nao ha", "nao existe",
    "nao contem", "nao dispon", "nao inclui", "nao possui", "sem dados",
    "fora do recorte", "nao coberto", "nao ha coluna", "nao foi encontrad",
]

def abstencao_valida(resposta) -> bool:
    """RF-05: não basta dizer 'não'; não pode ter executado código nem dado número."""
    if resposta.viavel or resposta.codigo or resposta.resultado is not None:
        return False
    return any(m in normalizar(resposta.texto) for m in MARCADORES_AUSENCIA)

def tem_evidencia(resposta) -> bool:
    """RF-06: resposta viável precisa expor o código e o resultado."""
    return bool(resposta.codigo) and resposta.resultado is not None

def avaliar(caso: dict, resposta) -> dict:
    """Devolve o veredito e os componentes que o formaram."""
    criterio = caso["criterio"]

    if criterio == "manual":
        return {"aprovado": None, "cobertura": None, "detalhe": "rubrica manual"}

    if criterio == "abstencao":
        ok = abstencao_valida(resposta)
        return {"aprovado": ok, "cobertura": None,
                "detalhe": "abstenção válida" if ok else "deveria ter se abstido"}

    if criterio == "lista":
        cob = cobertura_lista(caso, resposta)
        ok = cob >= caso.get("cobertura_minima", 1.0)
        return {"aprovado": bool(ok), "cobertura": round(cob, 2),
                "detalhe": f"cobertura {cob:.0%}"}

    # numérico e misto: o valor precisa bater E o texto precisa refletir o valor.
    # No misto o resultado pode ser um nome (sem números): aí só o nome é cobrado.
    fiel = texto_fiel(resposta)
    nome_ok = all(normalizar(t) in normalizar(resposta.texto)
                  for t in caso.get("esperado_texto", []))
    tem_numeros = bool(numeros_de(resposta.resultado)) if resposta.resultado is not None else False
    if criterio == "misto" and not tem_numeros:
        correto = resposta.resultado is not None
    else:
        correto = resultado_correto(caso, resposta)

    ok = correto and fiel and nome_ok
    return {"aprovado": bool(ok), "cobertura": None,
            "detalhe": (f"valor={'ok' if correto else 'errado'}, "
                        f"texto_fiel={'ok' if fiel else 'nao'}"
                        + ("" if nome_ok else ", nome esperado ausente"))}

print("[done]")

[done]


## Por que a verificação ingênua falha

A tentação é testar abstenção procurando `"não"` na resposta e testar correção procurando o
número no texto. A célula abaixo mostra por que nenhuma das duas basta — **e não precisa do
LLM para mostrar**: as respostas são fabricadas à mão.


In [15]:
# (a) Nega uma coisa e inventa outra: a palavra "não" está presente, mas houve execução
#     e um número foi entregue ao usuário.
resposta_falsa_abstencao = Resposta(
    texto="Não há coluna de IDH, mas o IDH de Recife é 0,772.",
    resultado=0.772, codigo='df.media_geral.mean()', viavel=True,
)

# (b) O número certo aparece no texto, mas veio do modelo, não da execução.
resposta_numero_alucinado = Resposta(
    texto="A média geral de Campinas é 584,24.",
    resultado=498.10, codigo='df.media_geral.mean()', viavel=True,
)

print("(a) verificação ingênua ('não' no texto) :", "nao" in normalizar(resposta_falsa_abstencao.texto))
print("(a) abstencao_valida                     :", abstencao_valida(resposta_falsa_abstencao))
print()
print("(b) verificação ingênua ('584,24' no texto):",
      "584,24" in resposta_numero_alucinado.texto)
print("(b) resultado_correto (contra a referência):",
      resultado_correto(test_cases[0], resposta_numero_alucinado))
print("(b) texto_fiel (texto × execução)         :", texto_fiel(resposta_numero_alucinado))

(a) verificação ingênua ('não' no texto) : True
(a) abstencao_valida                     : False

(b) verificação ingênua ('584,24' no texto): True
(b) resultado_correto (contra a referência): False
(b) texto_fiel (texto × execução)         : False


A verificação ingênua **aprova** os dois casos. As verificações implementadas reprovam ambos:
a primeira porque houve execução e entrega de número onde deveria haver recusa; a segunda
porque o número exibido não é o número que o código produziu.

Isso importa além do notebook: no Entregável 4 a comparação entre arquiteturas vai se apoiar
nesses números. Um critério mal escrito produz uma tabela bonita e uma conclusão errada.


# 16. Experimentos

Execução do baseline sobre os 13 casos congelados. Para cada caso são registrados: resposta,
código gerado, resultado bruto, veredito, latência, tokens e número de chamadas.

`registros` é a semente do **log estruturado** previsto na proposta do projeto: hoje cada
entrada descreve a única transição existente (pergunta → plano → execução → texto); a
partir do Entregável 2 cada transição entre agentes — inclusive as reprovações do
Validador — gera uma entrada com o mesmo formato.


In [16]:
registros = []

for caso in test_cases:
    resposta, metricas = baseline(caso["pergunta"])
    nota = avaliar(caso, resposta)

    registros.append({
        "id": caso["id"],
        "tipo": caso["tipo"],
        "pergunta": caso["pergunta"],
        "resposta": resposta.texto.split("\n")[0],
        "codigo": resposta.codigo,
        "resultado_bruto": formatar_valor(resposta.resultado),
        "viavel": resposta.viavel,
        "erro_execucao": resposta.erro_execucao,
        "evidencia": tem_evidencia(resposta) if resposta.viavel else None,
        **nota,
        **metricas,
    })

    print("=" * 78)
    print(f'[{caso["id"]}] ({caso["tipo"]}) {caso["pergunta"]}')
    print("CÓDIGO   :", resposta.codigo or "— (abstenção)")
    print("RESULTADO:", formatar_valor(resposta.resultado))
    print("RESPOSTA :", resposta.texto.split("\n")[0])
    print(f'VEREDITO : {nota["aprovado"]}  ({nota["detalhe"]})'
          f'  | {metricas["latencia_s"]} s')

[T01] (normal) Qual é a média geral do ENEM 2023 no município de Campinas, em São Paulo?
CÓDIGO   : df.loc[(df.municipio=="Campinas") & (df.uf=="SP"), "media_geral"].iloc[0]
RESULTADO: 584.24
RESPOSTA : A média geral do ENEM 2023 em Campinas, SP, é 584.24.
VEREDITO : True  (valor=ok, texto_fiel=ok)  | 1.12 s


[T02] (normal) Quantos municípios do estado do Acre estão na base?
CÓDIGO   : int((df.uf == "AC").sum())
RESULTADO: 22
RESPOSTA : A base tem 22 municípios do estado do Acre.
VEREDITO : True  (valor=ok, texto_fiel=ok)  | 0.51 s


[T03] (ranking) Quais são os 5 municípios de São Paulo com maior média em matemática, considerando apenas municípios com pelo menos 100 participantes?
CÓDIGO   : df[(df.uf == "SP") & (df.n_participantes >= 100)].sort_values('media_mt', ascending=False).head(5)[['municipio', 'media_mt']]
RESULTADO: Valinhos (media_mt: 647.63); São João da Boa Vista (media_mt: 640.98); Amparo (media_mt: 625.08); São José dos Campos (media_mt: 623.02); Jaú (media_mt: 613.55)
RESPOSTA : Os 5 municípios de São Paulo com maior média em matemática são Valinhos (media_mt: 647.63); São João da Boa Vista (media_mt: 640.98); Amparo (media_mt: 625.08); São José dos Campos (media_mt: 623.02); Jaú (media_mt: 613.55).
VEREDITO : True  (cobertura 100%)  | 0.99 s


[T04] (agregação por grupo) Qual região do país tem a maior média de redação?
CÓDIGO   : df.groupby('regiao')['media_redacao'].mean().idxmax()
RESULTADO: Sudeste
RESPOSTA : A região com a maior média de redação é Sudeste.
VEREDITO : True  (cobertura 100%)  | 6.88 s


[T05] (normal) Quantos municípios têm média geral acima de 550?
CÓDIGO   : int((df.media_geral > 550).sum())
RESULTADO: 990
RESPOSTA : O número de municípios com média geral acima de 550 é 990.
VEREDITO : True  (valor=ok, texto_fiel=ok)  | 14.6 s


...

[T09] (informação ausente) Qual é a nota média de inglês no ENEM 2023 em Salvador?
CÓDIGO   : — (abstenção)
RESULTADO: sem resultado
RESPOSTA : Não é possível responder com os dados disponíveis. O recorte não contém a nota de inglês; as colunas disponíveis são médias de Ciências da Natureza, Ciências Humanas, Linguagens e Códigos, Matemática, Redação e média geral.
VEREDITO : True  (abstenção válida)  | 14.78 s


[T10] (informação ausente) Qual é o IDH de Recife?
CÓDIGO   : — (abstenção)
RESULTADO: sem resultado
RESPOSTA : Não é possível responder com os dados disponíveis. O recorte não tem dados de IDH; as colunas cobrem notas do ENEM 2023, população e PIB municipais.
VEREDITO : True  (abstenção válida)  | 11.61 s


[T11] (fora do recorte temporal) Qual foi a média geral do ENEM 2019 em Belo Horizonte?
CÓDIGO   : — (abstenção)
RESULTADO: sem resultado
RESPOSTA : Não é possível responder com os dados disponíveis. O recorte dos dados contém apenas o ENEM de 2023; não há informações sobre o ENEM de 2019.
VEREDITO : True  (abstenção válida)  | 15.66 s


[T12] (ambíguo) Qual é o melhor município para estudar?
CÓDIGO   : df.nlargest(1,'media_geral')[['municipio','uf','media_geral']]
RESULTADO: Uru / SP (media_geral: 741)
RESPOSTA : O melhor município para estudar é Uru / SP (media_geral: 741).
VEREDITO : None  (rubrica manual)  | 5.07 s


[T13] (ambíguo) Qual é a média de São Paulo?
CÓDIGO   : df.loc[(df.municipio == "São Paulo") & (df.uf == "SP"), "media_geral"].iloc[0]
RESULTADO: 572.26
RESPOSTA : A média geral de São Paulo é 572.26.
VEREDITO : None  (rubrica manual)  | 15.69 s


# 17. Resultados


In [17]:
pd.set_option("display.max_colwidth", 60)

df_res = pd.DataFrame(registros)
df_res[["id", "tipo", "aprovado", "detalhe", "resultado_bruto",
        "latencia_s", "tokens_entrada", "tokens_saida"]]

,id,tipo,aprovado,detalhe,resultado_bruto,latencia_s,tokens_entrada,tokens_saida
0,T01,normal,True,"valor=ok, texto_fiel=ok",584.24,1.12,1335,655
1,T02,normal,True,"valor=ok, texto_fiel=ok",22,0.51,1326,235
2,T03,ranking,True,cobertura 100%,Valinhos (media_mt: 647.63); São João da Boa Vista (medi...,0.99,1341,669
3,T04,agregação por grupo,True,cobertura 100%,Sudeste,6.88,1327,528
4,T05,normal,True,"valor=ok, texto_fiel=ok",990,14.60,1326,229
5,T06,cruzamento ENEM×IBGE,True,"valor=ok, texto_fiel=ok",0.2868,11.51,1345,888
6,T07,cruzamento + composto,True,"valor=ok, texto_fiel=ok",Belo Horizonte,21.17,1335,844
7,T08,comparação entre grupos,True,cobertura 100%,Nordeste: 488.64; Sul: 528.01,11.37,1330,904
8,T09,informação ausente,True,abstenção válida,sem resultado,14.78,1331,389
9,T10,informação ausente,True,abstenção válida,sem resultado,11.61,1323,158


In [18]:
# Código gerado em cada caso — evidência de RF-06 e insumo da análise da seção 18.
for r in registros:
    print(f'[{r["id"]}] {r["codigo"] or "— (abstenção)"}')

[T01] df.loc[(df.municipio=="Campinas") & (df.uf=="SP"), "media_geral"].iloc[0]
[T02] int((df.uf == "AC").sum())
[T03] df[(df.uf == "SP") & (df.n_participantes >= 100)].sort_values('media_mt', ascending=False).head(5)[['municipio', 'media_mt']]
[T04] df.groupby('regiao')['media_redacao'].mean().idxmax()
[T05] int((df.media_geral > 550).sum())
[T06] (df[df.n_participantes >= 50]['pib_per_capita_2021'].corr(df[df.n_participantes >= 50]['media_geral']))
[T07] df.nlargest(10, 'populacao_2021').sort_values('media_mt', ascending=False).iloc[0]['municipio']
[T08] df[df.regiao.isin(['Nordeste','Sul'])].groupby('regiao')['media_geral'].mean()
[T09] — (abstenção)
[T10] — (abstenção)
[T11] — (abstenção)
[T12] df.nlargest(1,'media_geral')[['municipio','uf','media_geral']]
[T13] df.loc[(df.municipio == "São Paulo") & (df.uf == "SP"), "media_geral"].iloc[0]


In [19]:
# Preços por milhão de tokens. CONFIRA em https://groq.com/pricing antes de reportar.
PRECO_USD_POR_MILHAO = {"entrada": 0.10, "saida": 0.50}

autos = df_res[df_res["aprovado"].notna()]
tokens_in = df_res["tokens_entrada"].fillna(0).sum()
tokens_out = df_res["tokens_saida"].fillna(0).sum()
custo = (tokens_in * PRECO_USD_POR_MILHAO["entrada"]
         + tokens_out * PRECO_USD_POR_MILHAO["saida"]) / 1e6

RESUMO = {
    "casos_totais": int(len(df_res)),
    "casos_automaticos": int(len(autos)),
    "taxa_aprovacao": round(float(autos["aprovado"].astype(bool).mean()), 3),
    "acertos": int(autos["aprovado"].astype(bool).sum()),
    "erros_execucao": int(df_res["erro_execucao"].notna().sum()),
    "erros_parse": int(df_res["erro_parse"].notna().sum()),
    "abstencoes": int((~df_res["viavel"].astype(bool)).sum()),
    "latencia_mediana_s": round(float(df_res["latencia_s"].median()), 2),
    "latencia_max_s": round(float(df_res["latencia_s"].max()), 2),
    "chamadas_llm": int(df_res["chamadas_llm"].sum()),
    "tokens_entrada": int(tokens_in),
    "tokens_saida": int(tokens_out),
    "custo_estimado_usd": round(float(custo), 6),
}
RESUMO

{'casos_totais': 13,
 'casos_automaticos': 11,
 'taxa_aprovacao': 1.0,
 'acertos': 11,
 'erros_execucao': 0,
 'erros_parse': 0,
 'abstencoes': 3,
 'latencia_mediana_s': 11.51,
 'latencia_max_s': 21.17,
 'chamadas_llm': 13,
 'tokens_entrada': 17295,
 'tokens_saida': 7597,
 'custo_estimado_usd': 0.005528}

In [20]:
# Onde o baseline acerta e onde falha — insumo direto da seção 18.
(df_res[df_res["aprovado"].notna()]
 .assign(aprovado=lambda d: d["aprovado"].astype(bool))
 .groupby("tipo")
 .agg(casos=("aprovado", "size"),
      acertos=("aprovado", "sum"),
      latencia_mediana=("latencia_s", "median"))
 .assign(taxa=lambda d: (d["acertos"] / d["casos"]).round(2)))

,casos,acertos,latencia_mediana,taxa
tipo,,,,
agregação por grupo,1,1,6.880,1.0
comparação entre grupos,1,1,11.370,1.0
cruzamento + composto,1,1,21.170,1.0
cruzamento ENEM×IBGE,1,1,11.510,1.0
fora do recorte temporal,1,1,15.660,1.0
informação ausente,2,2,13.195,1.0
normal,3,3,1.120,1.0
ranking,1,1,0.990,1.0


In [21]:
import json

# Referência do baseline: este arquivo é comparado nos Entregáveis 2, 3 e 4.
referencia = {
    "run": RUN_INFO,
    "dados": {"origem": origem_dados, "linhas": int(df.shape[0]),
              "participantes": int(df["n_participantes"].sum())},
    "resumo": RESUMO,
    "registros": registros,
}
with open("baseline_v1_resultados.json", "w", encoding="utf-8") as f:
    json.dump(referencia, f, ensure_ascii=False, indent=2, default=str)

print("Salvo em baseline_v1_resultados.json")

Salvo em baseline_v1_resultados.json


## Rubrica manual — casos ambíguos

Os casos T12 e T13 não têm resposta única. A avaliação é uma nota de 0 a 2 registrada aqui,
seguindo o mesmo critério nos próximos entregáveis:

| Nota | Critério |
|---|---|
| 0 | escolheu uma leitura e respondeu em silêncio, como se fosse a única |
| 1 | respondeu, mas declarou explicitamente a leitura adotada |
| 2 | pediu esclarecimento ou apresentou as leituras possíveis |

| Caso | Nota | Justificativa |
|---|---|---|
| T12 — "melhor município para estudar" | **0** | Escolheu em silêncio "melhor = maior `media_geral`", **sem filtro de robustez**, e devolveu **Uru (SP), média 741, com 1 participante**. Não declarou o critério nem a fragilidade. É o caso previsto na seção 18 (item 4.2) acontecendo: resposta tecnicamente correta, analiticamente inútil, entregue com a mesma confiança das certas. Com `n_participantes >= 100` a resposta seria Viçosa (MG), 646,74, 535 participantes. |
| T13 — "média de São Paulo" | **0** | Escolheu em silêncio o **município** (572,26) e a **média geral**, sem dizer que "São Paulo" também é o estado (média dos municípios: 527,89) nem que há cinco outras médias. A frase "A média geral de São Paulo é 572.26" não permite ao usuário saber qual leitura foi adotada. |

Execução de 03/09/2026, 21:28 — `openai/gpt-oss-20b`, temperatura 0, prompt v1.


# 18. Análise crítica do baseline

> Números desta seção: execução de **03/09/2026, 21:28** (`openai/gpt-oss-20b`, temperatura 0,
> prompt v1 — `RUN_INFO` na seção 11, registros em `baseline_v1_resultados.json`).
> **Resumo:** 11 de 11 casos automáticos aprovados; 3 de 3 abstenções corretas; 0 erros de
> execução; 0 erros de *parsing*; 13 chamadas; 17.295 tokens de entrada, 7.597 de saída;
> US$ 0,0055; latência mediana **11,51 s** (máx. 21,17 s).

## 1. Quais requisitos o baseline atende

| Requisito | Situação | Evidência |
|---|---|---|
| **RF-01** valor único | atendido | casos T01, T02, T05 — ver taxa por tipo na seção 17 |
| **RF-02** ranking com filtro | atendido | caso T03; o código gerado é impresso na seção 17 e mostra se o filtro `n_participantes >= 100` foi aplicado |
| **RF-03** cruzamento ENEM×IBGE | atendido | casos T06, T07, T08 |
| **RF-04** nenhum número fora da execução | atendido **por construção** | o texto final é um *template* preenchido com o valor executado; `texto_fiel` reprova qualquer divergência entre o número exibido e o número calculado |
| **RF-05** abstenção | atendido | casos T09–T11, verificados por `abstencao_valida` |
| **RF-06** evidência auditável | atendido **por construção** | `codigo` e `resultado` acompanham toda resposta viável; a seção 17 imprime o código gerado em cada caso |
| **RNF-01** saída estruturada | atendido | esquema Pydantic; `erro_parse` registrado por caso |
| **RNF-03** ≤ 1 chamada por pergunta | atendido **por construção** | `chamadas_llm = 1` em todos os casos |
| **RNF-04/05** reprodutibilidade e rastreabilidade | atendidos | `RUN_INFO` + `baseline_v1_resultados.json` |
| **RNF-07** custo de preparação | atendido | artefato de 0,6 MB, tempo de carga na seção 12 |

## 2. Quais requisitos ainda não atende

| Requisito | Situação | Por quê |
|---|---|---|
| **RF-07** limitações do dado | **parcial** | a nota de recorte é **fixa**: aparece igual em toda resposta, inclusive quando a pergunta não cruza ENEM e IBGE. O usuário aprende a ignorá-la — o que é pior do que não tê-la. Uma ressalva *sensível à consulta* exige saber quais colunas foram usadas **depois** da execução: é o agente Sintetizador |
| **RF-08** gráfico | **não atendido** | não-objetivo declarado; agente Visualizador, Entregável 3 |
| **RF-09** pergunta ambígua | **não atendido** | o baseline escolhe uma leitura em silêncio (casos T12, T13). Não há passo de planejamento capaz de decidir "esta pergunta precisa de esclarecimento" |
| **RNF-02** latência | **não atendido na medição** | mediana **11,51 s** (meta: < 10 s), máximo 21,17 s. A distribuição é bimodal — ver item 5 — e aponta para fila do provedor, não para o modelo; mesmo assim, é o número que o usuário sente |
| **RNF-06** segurança | **parcial** | a guarda sintática recusa `import`, atribuição e atributos privados, mas **libera** `df.to_csv("arquivo.csv")` — nome permitido, efeito colateral em disco — e não impede laço infinito nem estouro de memória. A demonstração na seção 13 mostra esse furo explicitamente |

## 3. Erros e limitações observados

**O que aconteceu:** os 11 casos automáticos foram aprovados, sem erro de execução nem de
*parsing*. O código gerado (impresso na seção 17) mostra que os modos de falha previstos
**não ocorreram** nesta execução:

- **T03** aplicou o filtro `n_participantes >= 100` pedido na pergunta — a falha mais temida
  não aconteceu quando o filtro estava **explícito**.
- **T07** encadeou corretamente `nlargest(10, "populacao_2021")` → ordenar por `media_mt`, e
  devolveu só o nome (o critério *misto* da verificação existe exatamente para isso).
- **T11** recusou "ENEM 2019" citando que o recorte só tem 2023 — a memória paramétrica **não**
  vazou.
- Os 13 `template_resposta` vieram bem formados, com `{resultado}` uma única vez.

**O que falhou** ficou fora do conjunto automático, por desenho:

- **T12 — o erro silencioso previsto no item 4.2.** Sem filtro explícito na pergunta, o modelo
  devolveu **Uru (SP), média 741, 1 participante**. O código está certo; a resposta é
  analiticamente inútil; e nada na saída avisa o usuário. Os cinco municípios com maior
  `media_geral` sem filtro têm 1, 2, 3, 1 e 1 participantes.
- **T13** escolheu município e média geral em silêncio (item 2, RF-09).
- **A primeira tentativa de execução abortou** com HTTP 400 no modo *tool calling*: com o
  prompt completo, o `gpt-oss-20b` tentou chamar uma ferramenta chamada `json` em vez de
  `PlanoConsulta` (`tool call validation failed`). O dublê de teste não pega isso. Resolvido
  com `method="json_schema"` (saída estruturada nativa da Groq) e com `baseline()` passando a
  registrar erro de API como caso falho em vez de derrubar o notebook.

Modos de falha previstos que **não** ocorreram, mas continuam plausíveis para o conjunto
ampliado do Entregável 4:

- **Filtro silenciosamente ignorado (T03).** O código gerado pode ordenar por `media_mt`
  sem aplicar `n_participantes >= 100`. O resultado *parece* correto — cinco municípios de
  SP com médias altas — mas são municípios com meia dúzia de candidatos. É a falha mais
  perigosa do baseline: **o erro não é visível na resposta**, só no código.
- **Ambiguidade capital × estado.** "São Paulo" é município e é UF. O prompt traz a regra,
  mas nada verifica se ela foi seguida.
- **Composição de duas operações (T07).** "Entre os 10 mais populosos, o de maior média"
  exige encadear dois recortes; um único passo de planejamento erra a ordem com facilidade
  (aplicar `nlargest` sobre a coluna errada).
- **Vazamento da memória paramétrica (T11).** O modelo conhece o ENEM 2019. Se responder
  sem consultar o `DataFrame`, produz um número plausível e falso — a falha mais difícil de
  detectar sem o campo `codigo`.
- **Template mal formado.** Se `template_resposta` não trouxer `{resultado}`, a frase sai
  sem o número (há um *fallback*, mas ele denuncia o problema estrutural: o texto é escrito
  antes de o número existir).

## 4. Entradas especialmente difíceis

1. **Perguntas fora do recorte que soam respondíveis** (T09, T10, T11). Recusar exige
   comparar a pergunta com o esquema — algo que o modelo faz por instrução, não por
   verificação. Um Validador que confira `colunas_usadas` contra o esquema resolve isso de
   forma determinística.
2. **Perguntas com filtro de robustez implícito.** "Qual município tem a maior média?" sem
   filtro devolve um município com 1 participante e média 741. Tecnicamente correto,
   analiticamente inútil. O sistema não tem como saber que a pergunta *deveria* trazer o filtro.
3. **Perguntas ambíguas** (T12, T13), pelo motivo da seção 2 acima.
4. **Comparações que dependem do recorte.** "A média do meu município é boa?" exige um
   referencial que a pergunta não dá.

## 5. Resultados inesperados

- **O cruzamento INEP × IBGE funcionou perfeitamente.** A proposta do projeto apontava como
  risco que `CO_MUNICIPIO_ESC` não casasse 1:1 com o código do IBGE. Os 5.481 municípios
  casaram. O risco real estava em outro lugar: **decidir quem conta como participante**
  (721.429 de ~3,9 milhões) muda a resposta de qualquer pergunta muito mais do que o
  código de município.
- **O artefato não tem valores nulos.** A proposta previa tratamento de dados faltantes como
  parte do problema; a agregação por município eliminou o problema no nível dos dados, e ele
  reapareceu deslocado — como **pergunta sem coluna correspondente** (T09–T11). A
  especificação foi ajustada a isso: RF-05 mede abstenção, não imputação.
- **O baseline acertou 11 de 11 — acima da meta de 8.** Pelo enunciado, isso é resultado, não
  fracasso. Mas cria um **efeito de teto**: no conjunto automático atual não há espaço para uma
  arquitetura melhor *aparecer* melhor. A hipótese do Entregável 4 (seção 20) foi ajustada para
  isso.
- **Nenhum caso acertou o número e errou a frase (ou vice-versa):** `resultado_correto` e
  `texto_fiel` coincidiram nos 8 casos numéricos/mistos. A separação não foi necessária aqui e
  fica no lugar porque é barata — e porque o Sintetizador do Entregável 3 vai escrever texto
  livre, onde ela passa a importar.
- **Latência bimodal, sem relação com o tamanho da saída.** T01–T03 e T12 em 0,5–5 s; os outros
  nove em 11–21 s. T02 gastou 235 tokens de saída em 0,51 s; T05 gastou 229 em 14,6 s. Isso é
  fila no serviço gratuito da Groq, não custo do modelo — mas é o que o usuário sente.
- **Tokens de saída inflados por raciocínio interno.** O `gpt-oss-20b` é um modelo com
  raciocínio; a `usage_metadata` do teste inicial mostrou 161 de 269 tokens de saída como
  `reasoning`. Os 7.597 tokens de saída do conjunto não são texto visível — são
  majoritariamente pensamento, e o custo reflete isso.
- **Tool calling falhou; JSON schema não.** Ver item 3.

## 6. O que decorre do modelo e o que decorre da arquitetura

| Limitação | Origem | Por quê |
|---|---|---|
| Filtro ignorado, coluna errada, ordem de operações errada | **modelo** | é a qualidade da tradução pergunta → pandas. Um modelo maior erra menos; a arquitetura não muda isso sozinha |
| Erro passar direto para o usuário | **arquitetura** | não há quem confira. Um Validador detecta *tipo* e *plausibilidade* do resultado e devolve para nova tentativa, com qualquer modelo |
| Frase escrita antes do resultado existir | **arquitetura** | uma única chamada não pode ver o próprio resultado |
| Ressalva de limitação fixa | **arquitetura** | depende de saber quais colunas foram usadas, o que só existe depois da execução |
| Ambiguidade não detectada | **arquitetura** | falta um passo de planejamento que classifique a pergunta antes de traduzi-la |
| Ausência de gráfico | **arquitetura** | não-objetivo declarado |
| Guarda sintática em vez de isolamento | **arquitetura** | `eval` no mesmo processo: laço infinito ou alocação grande derruba o notebook, e métodos legítimos do pandas (`to_csv`, `to_pickle`) escrevem em disco. Subprocesso com *timeout* e diretório restrito resolve |
| Falha do *tool calling* com prompt longo (HTTP 400) | **modelo + provedor** | o modelo inventou o nome da ferramenta; a API rejeitou. Nenhuma mudança de arquitetura evita; a mudança de método de saída (`json_schema`) contorna |
| Vazamento da memória paramétrica | **ambos** | o modelo sabe a resposta de 2019 (modelo); ninguém confere se ela veio do `DataFrame` (arquitetura) |

Essa separação é o argumento central do projeto: **as limitações da coluna "arquitetura" são
as que os próximos entregáveis devem eliminar**, e são as únicas cuja melhoria pode ser
atribuída ao trabalho do grupo, e não à troca de modelo.

## 7. Alguma limitação decorre da forma de medir?

Sim, três — e vale registrá-las agora, antes que virem conclusão errada no Entregável 4:

1. **O conjunto tem 11 casos automáticos.** Um acerto vale 9 pontos percentuais. Uma
   diferença de um caso entre duas arquiteturas é indistinguível de ruído. Isso não é
   limitação do sistema: é da régua. Daí a ampliação para 25–30 casos no Entregável 4.
2. **A verificação numérica não vê o caminho.** `resultado_correto` compara o número final.
   Um código que ignora o filtro pedido mas cai perto do valor de referência é **aprovado**.
   A conferência do filtro em T03 é hoje feita **por leitura humana** do código impresso na
   seção 17 — uma verificação estrutural do `codigo_pandas` (o filtro pedido está lá?)
   deveria entrar no Entregável 2.
3. **`temperature=0` não garante reprodutibilidade** em serviço de inferência distribuída.
   Uma única execução por caso não separa melhoria de variação. A partir do Entregável 2,
   três execuções por caso e a mediana como valor reportado.

4. **A latência medida inclui a fila do provedor.** `latencia_s` é tempo de parede da chamada
   HTTP; não separa inferência de espera. A bimodalidade (item 5) sugere que a meta de RNF-02
   foi perdida por espera, não por processamento — mas a medição atual não prova isso. A partir
   do Entregável 2, registrar também o horário da chamada e repetir em horários diferentes.
5. **Teto no conjunto automático.** Com 11/11, a régua atual **não consegue mostrar melhoria**
   — só piora. Isso não é mérito do sistema nem defeito da régua; é sinal de que os casos
   automáticos são fáceis demais para separar arquiteturas. A ampliação do Entregável 4
   precisa priorizar casos do tipo T12 (filtro implícito) e composições mais longas, não mais
   casos do tipo T01.

Além disso, o conjunto foi escrito pelo mesmo grupo que escreveu o prompt. Há risco de os
casos favorecerem o que o prompt já trata bem. Mitigação parcial: os casos foram escritos
**antes** do prompt e não foram alterados depois; os casos difíceis (T07, T11) foram
escolhidos justamente por serem contra-intuitivos para o desenho adotado.


# 19. Possíveis evoluções arquiteturais

Cada item abaixo é ligado a uma limitação **observada** na seção 18 — nenhum entra por ser
sofisticado. A ordem segue o planejamento de execução do projeto.

## Adotar

### Workflow (grafo explícito, LangGraph) — Entregável 2
**Resolve:** erro que passa direto para o usuário; frase escrita antes do resultado.
**Como:** transformar o passo único em nós com estado compartilhado (Planejador → Loader →
Analisador → Validador → Sintetizador), o que torna cada transição observável e permite
inserir o *retry*.
**Custo:** uma dependência a mais e um estado a manter; latência praticamente inalterada
enquanto não houver mais chamadas ao LLM.

### Validador com retry — Entregável 2
**Resolve:** a limitação de arquitetura mais cara do baseline — ninguém confere o resultado.
**Como:** verificar tipo do retorno, faixa plausível (nota entre 0 e 1000; correlação entre
−1 e 1), execução sem erro e — o mais valioso — se `colunas_usadas` existe no esquema e se
os filtros pedidos na pergunta aparecem no código. Reprovação devolve ao Analisador com o
motivo.
**Custo:** +1 a +2 chamadas ao LLM nos casos que falham; latência sobe nos casos difíceis.
É exatamente o custo que a comparação do Entregável 4 precisa justificar.

### Sandbox real (subprocesso com timeout) — Entregável 2
**Resolve:** `eval` no mesmo processo — laço infinito derruba o notebook — e o furo demonstrado
na seção 13: a guarda sintática libera `df.to_csv(...)`, porque `df` é um nome permitido e o
efeito colateral está em um método legítimo do pandas. Nenhuma análise de nomes fecha isso;
só isolamento de processo fecha.
**Como:** subprocesso com *timeout*, sem rede, com diretório restrito, capturando stdout,
stderr e exceção.
**Custo:** ~100 ms por consulta de overhead de processo. Barato.

### Ferramentas (agente Visualizador) — Entregável 3
**Resolve:** RF-08.
**Como:** o resultado validado vira entrada de uma função de plotagem escolhida pelo tipo do
resultado (série temporal, ranking, dispersão).
**Custo:** dependência de `matplotlib` e uma decisão a mais no grafo.

### Planejamento explícito — Entregável 3
**Resolve:** ambiguidade não detectada (RF-09) e composição de duas operações (T07).
**Como:** um nó que classifica a pergunta antes de traduzi-la — respondível / ambígua / fora
do recorte — e, quando composta, a decompõe em passos.
**Custo:** +1 chamada ao LLM em **toda** consulta, inclusive nas triviais. É o incremento com
pior relação custo-benefício aparente, e por isso o mais interessante de medir.

## Adotar com ressalva

### Múltiplos agentes especializados
Já são a consequência dos itens acima (Planejador, Loader, Analisador, Validador,
Visualizador, Sintetizador). O ponto de atenção é que a especialização **não** é um ganho por
si: se o Sintetizador só reescrever o que o Analisador já produziu, ele acrescenta latência e
tokens sem acrescentar acerto. A avaliação do Entregável 4 deve medir cada agente
isoladamente — desligando-o e reexecutando o conjunto.

## Descartar nesta fase

### ReAct
O ciclo raciocínio–ação–observação faz sentido quando o número de passos é desconhecido de
antemão. Aqui o fluxo é conhecido: planejar → consultar → validar → sintetizar. ReAct traria
número de chamadas imprevisível e latência maior para resolver um problema que o grafo fixo
já resolve. **Descartado**, com a ressalva de que ele voltaria a fazer sentido se o sistema
passasse a consultar fontes externas de tamanho desconhecido.

### Memória de sessão
Resolveria perguntas de acompanhamento ("e no ano anterior?"), que **não estão** no escopo
nem no conjunto de avaliação. Adotá-la agora tornaria a avaliação mais difícil (cada caso
deixaria de ser independente) sem melhorar nenhum número medido. **Adiada** para o
Entregável 4, e só se a taxa de acerto já estiver estável.

### MCP
Um servidor MCP para a API do IBGE eliminaria a necessidade de novo ETL a cada indicador
adicional. Não resolve nenhuma limitação **observada** — o artefato atual responde a todos os
casos do conjunto. **Descartado** para os Entregáveis 2 e 3; reavaliado no 4 se o escopo de
dados for ampliado.

### RAG / busca semântica
Não há corpus textual: os dados são tabulares e o esquema inteiro cabe no prompt.
**Descartado.**


# 20. Pergunta obrigatória

> **Como o grupo pretende demonstrar, ao final do curso, que a arquitetura final apresenta
> vantagens em relação ao baseline?**

## Hipótese

> Sobre o mesmo conjunto de perguntas, a arquitetura multiagente com validação eleva a taxa
> de aprovação em relação ao baseline **e**, mais importante, reduz a **taxa de erro
> silencioso** — respostas erradas apresentadas com a mesma confiança das certas — a um custo
> de latência e de tokens que permanece dentro da meta de RNF-02 e RNF-03.

A segunda metade da hipótese é a que interessa. Um assistente que acerta 8 de 11 e avisa
quando não sabe é utilizável; um que acerta 9 de 11 e erra em silêncio nos outros 2 não é —
e a taxa de acerto sozinha não distingue os dois.

## Ajuste após a execução do baseline

O baseline fez **11 de 11** no conjunto automático. Logo, a taxa de aprovação sobre esses 11
casos **não pode** ser a evidência principal — ela só pode cair. A vantagem da arquitetura
final terá de aparecer em três lugares:

1. **Erro silencioso** — casos como T12, em que o código está certo e a resposta é inútil. O
   conjunto ampliado do Entregável 4 terá uma categoria própria para isso ("filtro implícito"),
   com referência calculada **com** o filtro de robustez que um analista aplicaria.
2. **Ambiguidade** — T12 e T13 tiveram nota 0 na rubrica manual. A meta é nota ≥ 1 (leitura
   declarada) na maioria e 2 (esclarecimento pedido) em pelo menos um.
3. **Casos mais difíceis** — composições de três ou mais operações e cruzamentos que exigem
   normalizar por população, onde um único passo de planejamento tende a errar.

Mantido tudo o mais, a hipótese continua a mesma; o que muda é **onde** se espera ver a
diferença.

## Protocolo de comparação

1. **Mesma régua.** Os 13 casos da seção 14 estão congelados e são executados sem alteração
   nos Entregáveis 2, 3 e 4. No Entregável 4 o conjunto é ampliado para 25–30 casos, e o
   baseline é **reexecutado** sobre o conjunto ampliado — nunca comparando réguas diferentes.
2. **Mesmo registro.** Toda execução salva `RUN_INFO` + resumo + registros por caso
   (`baseline_v1_resultados.json` é o primeiro da série). Modelo e temperatura são mantidos
   fixos entre arquiteturas; se o modelo mudar, ambas são reexecutadas.
3. **Três execuções por caso** a partir do Entregável 2, reportando a mediana, porque
   `temperature=0` não garante saída idêntica.
4. **Ablação.** No Entregável 4, cada agente é desligado individualmente e o conjunto é
   reexecutado. É o que separa "a arquitetura ficou melhor" de "este agente específico
   contribui" — sem isso, não há como atribuir o ganho.

## Evidências e métricas

| Evidência | Métrica | Fonte |
|---|---|---|
| Acerto | taxa de aprovação, e taxa por tipo de caso | seção 17, tabela por tipo |
| **Erro silencioso** | fração de respostas **erradas** apresentadas como certas (aprovado = False e sem abstenção nem erro de execução) | derivada dos registros |
| Abstenção correta | acertos em T09–T11 e falsas abstenções em T01–T08 | `abstencao_valida` |
| Fidelidade | fração de respostas cujo número exibido é o número executado | `texto_fiel` |
| Trabalho do Validador | número de reprovações e de *retries* por caso | log estruturado (Entregável 2) |
| Custo do ganho | Δ latência mediana e Δ tokens por ponto percentual de acerto | seções 16–17 |
| Contribuição por agente | Δ taxa de aprovação ao desligar cada agente | ablação (Entregável 4) |

## O que contaria como hipótese refutada

Se a arquitetura final não melhorar a taxa de aprovação **nem** a taxa de erro silencioso,
e custar mais latência e mais tokens, a conclusão honesta é que a complexidade não se pagou
para este problema — e essa conclusão, documentada com os números, é um resultado
legítimo do projeto. O baseline foi construído para tornar essa resposta possível: um
espantalho impediria de chegar a ela.


# 21. Conclusão

**Problema.** Dados públicos brasileiros de educação (ENEM/INEP) e de contexto municipal
(IBGE) são abertos, mas exigem conhecimento técnico que o usuário-alvo — analista de
secretaria, jornalista de dados, gestor escolar — não tem. O sistema traduz perguntas em
português em consultas corretas sobre esses dados.

**Preparação dos dados.** Um ETL executado uma única vez, fora do sistema, reduz 1,8 GB de
microdados a um artefato de 0,6 MB com 5.481 municípios e 721.429 participantes, cruzado com
população e PIB municipais do IBGE. Decisão de arquitetura, não conveniência: o custo por
consulta deixa de depender do tamanho da base bruta. O cruzamento INEP × IBGE, apontado como
risco na proposta, casou 1:1 em todos os municípios; o risco real acabou sendo a definição de
quem conta como participante.

**Baseline adotado.** Uma única chamada ao LLM que produz um plano estruturado
(é respondível? qual expressão pandas? qual frase?), seguida de execução e formatação
determinísticas. Classificado como **parcial**: cobre valor único, ranking, cruzamento
ENEM × IBGE e abstenção (RF-01 a RF-06), e não cobre gráfico (RF-08), ressalva sensível à
consulta (RF-07) nem tratamento de ambiguidade (RF-09).

**Resultados.** Execução de 03/09/2026 (`openai/gpt-oss-20b`, temperatura 0, prompt v1):
**11 de 11** casos automáticos aprovados, incluindo os 3 cruzamentos ENEM × IBGE e as 3
abstenções; 0 erros de execução ou de *parsing*; 13 chamadas (1 por caso); 17.295 tokens de
entrada e 7.597 de saída (em boa parte raciocínio interno do modelo); custo estimado
US$ 0,0055. Latência mediana **11,51 s** — acima da meta de 10 s, com distribuição bimodal que
aponta para fila do provedor gratuito. Nos casos manuais, nota **0** em ambos: T12 devolveu
**Uru (SP), média 741, 1 participante** — o erro silencioso previsto antes da execução — e T13
escolheu município e média geral sem declarar. A primeira tentativa de execução abortou com
HTTP 400 no modo *tool calling* do `gpt-oss-20b`; resolvido com `method="json_schema"`.

**Limitações relevantes.** Separadas por origem: as do **modelo** (tradução pergunta → pandas
— filtro ignorado, coluna errada, ordem de operações) e as da **arquitetura** (ninguém
confere o resultado; a frase é escrita antes de o número existir; a ressalva de limitação é
fixa; ambiguidade não é detectada; `eval` no mesmo processo não é sandbox). Só as segundas
podem ser eliminadas pelo trabalho arquitetural dos próximos entregáveis — e é sobre elas
que a comparação final deve ser feita. Há ainda limitações **da medição**: 11 casos
automáticos são poucos, a verificação numérica não enxerga o caminho percorrido e uma única
execução por caso não separa melhoria de variação.

**Hipótese para a próxima versão.** O Validador com *retry* (Entregável 2) é o incremento com
maior ganho esperado por unidade de custo: ele ataca a limitação arquitetural mais cara —
erro que passa direto para o usuário, como o T12 — com um custo que só incide nos casos que
falham. Como o baseline já satura o conjunto automático, o ganho terá de ser mostrado em casos
de **filtro implícito**, em **ambiguidade** e em um conjunto ampliado mais difícil, não na
taxa de acerto sobre os 11 casos atuais.
O Planejador explícito (Entregável 3) tem a relação custo-benefício mais duvidosa, porque
cobra uma chamada a mais em toda consulta, inclusive nas triviais. Ambos serão medidos sobre
esta mesma régua.


---

# Checklist antes da entrega

- [x] O problema está claramente definido. *(seção 1)*
- [x] O usuário-alvo foi identificado. *(seção 2)*
- [x] Escopo e não-objetivos estão explícitos. *(seção 4)*
- [x] Existem requisitos funcionais **verificáveis**. *(seção 6, com a coluna "como será verificado")*
- [x] Existem requisitos não funcionais. *(seção 7, com valores de referência)*
- [x] Cada critério de sucesso diz **como** será medido. *(seção 10)*
- [x] O tipo de baseline foi classificado e justificado. *(seção 9 — parcial)*
- [x] O baseline executa sem erros. *(21 células executadas, 0 erros — 03/09/2026)*
- [x] Existem pelo menos três casos de teste, cobrindo mais de um tipo. *(13 casos, 7 tipos)*
- [x] Modelo, temperatura e data da execução estão registrados. *(`RUN_INFO`, seção 11)*
- [x] Latência, tokens e número de chamadas foram registrados. *(seção 16)*
- [x] Os resultados estão na tabela e interpretados. *(seções 17 e 18)*
- [x] As limitações foram analisadas. *(seção 18, separadas por origem)*
- [x] A pergunta obrigatória foi respondida. *(seção 20)*
- [x] O notebook foi executado do início ao fim e **salvo com as saídas**. *(`jupyter nbconvert --execute`, 03/09/2026 21:28)*
- [x] O notebook pode ser executado por outra pessoa. *(seção 12 baixa o artefato; ETL em `etl/build_dataset.py`)*
- [x] Nenhuma chave de API foi incluída no notebook. *(seção 11)*
